# Nebula V5 — keep learning, keep the best model

Import into a **private Kaggle notebook**, enable **Internet** and **GPU T4 x2**, and stop older Nebula trainers first. Enable the existing `NEBULA_TRAINER_TOKEN` and `NEBULA_HF_TOKEN` secrets. Run all. No Gemini, Groq, or Colab GPU is needed.

V5 continues the strongest saved V4 trial, rather than its rejected latest trial or the original model. Original models remain intact. It adds original short conversations, follow-up dialogue, and exact arithmetic with complete terminating answers. Source notices for the existing public data remain in the preserved dataset manifest. Known familiar questions are explicit rehearsal, not unseen evaluation.

Every 500 updates it evaluates final and lowest-loss checkpoints on frozen V4 tasks, six new conversation checks, and the same validation sample. Research selection permits modest category tradeoffs when overall results improve. It retains the best research head, rotates practice on plateaus, and **does not stop after three rejected trials**. These tiny diagnostics do not establish general chat quality.

After the updated API is deployed, currently opted-in account contributions provide coarse topic counts. At most one round in four follows those topics. The trainer uses owned, verified practice for those topics; raw chats, names, IDs, and the model's unchecked answers are not uploaded or learned as facts. If the feed is unavailable or too small, training continues on the prepared curriculum. Account opt-outs and deleted contributions are checked again on each feed request.

Each completed round saves weights, reports, and a resume manifest to your **private** Hugging Face repository. A failed backup stops the run. Uploaded weights omit optimizer state to reduce storage; resumed rounds start fresh AdamW. Saved best and latest models are separate. The notebook never changes the public release.

Run for up to eight hours or 100 backup rounds per session, then rerun the same notebook and run name to continue. Kaggle GPU quotas and session limits still apply. Checkpoint history consumes storage; 100 latest snapshots of an 8.2M model are about 3.3 GB, plus improved-best snapshots and reports. Do not run duplicate notebooks. A refresh or disconnect can lose work since the last `checkpointBackupConfirmed` message. Backups restore only successfully uploaded rounds.


In [ ]:
import os,sys,subprocess
from pathlib import Path
import torch
from kaggle_secrets import UserSecretsClient
subprocess.run([sys.executable,'-m','pip','install','--quiet','huggingface_hub>=0.34,<2'],check=True)
def secret(name):
    try:value=UserSecretsClient().get_secret(name)
    except Exception:raise RuntimeError(f'Add and enable Kaggle secret {name}; check Internet is on.') from None
    if not isinstance(value,str) or not value.strip():raise RuntimeError(f'Empty Kaggle secret: {name}')
    return value.strip()
os.environ['NEBULA_WORKER_TOKEN']=secret('NEBULA_TRAINER_TOKEN')
os.environ['NEBULA_HF_TOKEN']=secret('NEBULA_HF_TOKEN')
os.environ['NEBULA_HF_REPO']='iiidevv/nebula-research'
os.environ['NEBULA_RESEARCH_API']='https://api.scriptnovaa.com/api/nebula/research'
RUN_NAME='nebula-v5'  # Keep this name to resume saved progress.
HOURS=8
MAX_ROUNDS=100
STEPS_PER_ROUND=500
project=Path('/kaggle/working/nebula-v5/api')
project.mkdir(parents=True,exist_ok=True)
if not torch.cuda.is_available():raise RuntimeError('Enable a Kaggle GPU accelerator.')
print({'GPU':torch.cuda.get_device_name(0),'availableGPUs':torch.cuda.device_count(),'publicAutoUpdate':False})


In [ ]:
sources={'model/nebula.py': '"""Nebula\'s small, randomly initialized causal Transformer. No pretrained weights."""\nimport json\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nimport torch\nfrom torch import nn\n\nBOS, EOS, SYSTEM, USER, ASSISTANT, PAD = range(256, 262)\nVOCAB = 262\nROLES = {"system": SYSTEM, "user": USER, "assistant": ASSISTANT}\n\n@dataclass\nclass Config:\n    width: int = 128\n    heads: int = 4\n    layers: int = 2\n    context: int = 512\n\ndef config_for_preset(preset):\n    presets = {\n        \'smoke\': Config(width=64,heads=2,layers=1,context=256),\n        \'cpu\': Config(),\n        \'cpu-medium\': Config(width=192,heads=6,layers=4,context=1024),\n        \'cpu-large\': Config(width=256,heads=8,layers=6,context=1024),\n        \'gpu\': Config(width=384,heads=8,layers=6,context=2048),\n    }\n    if preset not in presets: raise ValueError(\'Unknown model preset\')\n    return presets[preset]\n\ndef encode(messages, answer_prefix=False):\n    tokens, learn = [BOS], [False]\n    for message in messages:\n        role = message["role"]\n        tokens.append(ROLES[role]); learn.append(False)\n        content = list(message["content"].encode("utf-8"))\n        tokens.extend(content); learn.extend([role == "assistant"] * len(content))\n        tokens.append(EOS); learn.append(role == "assistant")\n    if answer_prefix:\n        tokens.append(ASSISTANT); learn.append(False)\n    return tokens, learn\n\ndef pack_prompt(messages, context):\n    systems = [m for m in messages if m[\'role\'] == \'system\']\n    history = [m for m in messages if m[\'role\'] != \'system\']\n    if not history or history[-1][\'role\'] != \'user\': raise ValueError(\'A user question is required\')\n    prefix, _ = encode(systems)\n    selected = []\n    # Keep every system instruction and the full latest question; never silently cut rules.\n    available = context - len(prefix) - 2\n    for message in reversed(history):\n        chunk = encode([message])[0][1:]\n        if len(chunk) > available:\n            if not selected: raise ValueError(\'Rules and question exceed the model context\')\n            break\n        selected.insert(0, chunk); available -= len(chunk)\n    return prefix + [token for chunk in selected for token in chunk] + [ASSISTANT]\n\nclass Nebula(nn.Module):\n    def __init__(self, config):\n        super().__init__(); self.config = config\n        self.token = nn.Embedding(VOCAB, config.width)\n        self.position = nn.Embedding(config.context, config.width)\n        self.blocks = nn.ModuleList(nn.TransformerEncoderLayer(config.width, config.heads, config.width * 4, dropout=0.0, activation="gelu", batch_first=True, norm_first=True) for _ in range(config.layers))\n        self.norm = nn.LayerNorm(config.width)\n        self.output = nn.Linear(config.width, VOCAB, bias=False)\n        self.output.weight = self.token.weight\n        self.register_buffer(\'_causal_mask\',torch.ones(config.context,config.context,dtype=torch.bool).triu(1),persistent=False)\n        self.apply(self._initialize)\n\n    @staticmethod\n    def _initialize(module):\n        if isinstance(module, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(module.weight, std=0.02)\n            if isinstance(module, nn.Linear) and module.bias is not None: nn.init.zeros_(module.bias)\n\n    def forward(self, tokens):\n        length = tokens.shape[1]\n        if length > self.config.context: raise ValueError("Context is too long")\n        value = self.token(tokens) + self.position(torch.arange(length, device=tokens.device))\n        causal = self._causal_mask[:length,:length]\n        for block in self.blocks:\n            if self.training and getattr(self,\'_activation_checkpointing\',False) and torch.is_grad_enabled():\n                from torch.utils.checkpoint import checkpoint\n                value=checkpoint(block,value,src_mask=causal,use_reentrant=False)\n            else:value=block(value,src_mask=causal)\n        return self.output(self.norm(value))\n\n    @torch.inference_mode()\n    def reply(self, messages, max_tokens=160, temperature=0.7):\n        self.eval()\n        encoded = pack_prompt(messages, self.config.context)\n        device = next(self.parameters()).device\n        tokens = torch.tensor([encoded], device=device)\n        result = []\n        for _ in range(min(max_tokens, self.config.context - len(encoded))):\n            logits = self(tokens)[:, -1, :].clone()\n            end = logits[:, EOS].clone(); logits[:, BOS:] = float(\'-inf\'); logits[:, EOS] = end\n            if temperature == 0: next_token = logits.argmax(-1, keepdim=True)\n            else: next_token = torch.multinomial(torch.softmax(logits / temperature, -1), 1)\n            value = int(next_token.item())\n            if value == EOS: break\n            result.append(value); tokens = torch.cat([tokens, next_token], dim=1)\n        return bytes(result).decode(\'utf-8\', errors=\'replace\').strip()\n\ndef save(model, path, metadata, training_state=None):\n    if any(not torch.isfinite(parameter).all() for parameter in model.parameters()):\n        raise ValueError(\'Checkpoint contains non-finite weights; existing saved weights were not replaced\')\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(\'.tmp\')\n    payload={\'format\': 1, \'config\': asdict(model.config), \'state\': model.state_dict(), \'metadata\': metadata}\n    if training_state is not None: payload[\'trainingState\']=training_state\n    torch.save(payload, temporary)\n    temporary.replace(path)\n\ndef load(path):\n    data = torch.load(path, map_location=\'cpu\', weights_only=True)\n    if data.get(\'format\') != 1: raise ValueError(\'Unsupported model checkpoint\')\n    model = Nebula(Config(**data[\'config\'])); model.load_state_dict(data[\'state\']); model.eval()\n    if any(not torch.isfinite(parameter).all() for parameter in model.parameters()):\n        raise ValueError(\'Checkpoint contains non-finite weights; resume from a healthy best checkpoint\')\n    return model, data.get(\'metadata\', {})\n', 'model/train.py': 'import argparse\nimport hashlib\nimport json\nimport random\nimport time\nimport itertools\nimport math\nfrom contextlib import nullcontext\nfrom pathlib import Path\nimport torch\nfrom torch.nn import functional as F\nfrom nebula import Config, Nebula, PAD, encode, load, save, config_for_preset\nfrom dataflow import read_jsonl, eligible\nfrom rules import with_owner_rules\n\nclass NumericalFailure(RuntimeError): pass\n\ndef examples(rows, context, include_owner_rules=False, objective=\'assistant\'):\n    result = []\n    for row in rows:\n        messages = row[\'messages\']\n        if objective == \'language\':\n            # Language pretraining uses continuous serialized text chunks, with no\n            # invented end-of-answer tokens at chunk boundaries. Chat tuning below\n            # uses complete replies instead.\n            prefix=encode(with_owner_rules([]) if include_owner_rules else [])[0]\n            body=encode(messages)[0][1:]\n            room=context+1-len(prefix)\n            if room<1: raise ValueError(\'Owner rules exceed model context\')\n            for offset in range(0,len(body),room):\n                tokens=prefix+body[offset:offset+room]\n                x=torch.tensor(tokens[:-1]); y=torch.tensor([-100]*(len(prefix)-1)+tokens[len(prefix):])\n                result.append((x,y))\n            continue\n        wrapped = with_owner_rules(messages) if include_owner_rules else messages\n        windows = [wrapped]\n        if len(encode(wrapped)[0]) > context + 1:\n            windows = []\n            for end in range(2,len(messages)+1,2):\n                for start in range(0,end,2):\n                    candidate=with_owner_rules(messages[start:end]) if include_owner_rules else messages[start:end]\n                    if len(encode(candidate)[0]) <= context+1:\n                        windows.append(candidate); break\n        for window in windows:\n            tokens, learn = encode(window)\n            x = torch.tensor(tokens[:-1]); y = torch.tensor([v if mask else -100 for v, mask in zip(tokens[1:], learn[1:])])\n            if (y != -100).any(): result.append((x, y))\n    return result\n\ndef partition_rows(rows, seed):\n    """Split original dialogues before creating windows, keeping duplicates together."""\n    training=[]; validation=[]\n    for row in rows:\n        identifier=hashlib.sha256(json.dumps(row[\'messages\'],sort_keys=True).encode()).hexdigest()\n        # Owned curriculum variants of one scenario stay in the same partition.\n        group=row.get(\'splitGroup\') if row.get(\'source\',{}).get(\'kind\') in [\'seed\',\'public\',\'teacher\'] else None\n        split_key=\'group:\'+group if isinstance(group,str) and group else identifier\n        score=int(hashlib.sha256((str(seed)+\':\'+split_key).encode()).hexdigest()[:8],16)\n        (validation if score%10==0 else training).append(row)\n    if not training or not validation: raise ValueError(\'Grouped split requires enough distinct conversations\')\n    return training,validation\n\ndef bucket_batches(data,batch_size,rng):\n    ordered=sorted(data,key=lambda pair:len(pair[0])); batches=[]\n    for offset in range(0,len(ordered),max(batch_size*32,32)):\n        bucket=ordered[offset:offset+max(batch_size*32,32)]; rng.shuffle(bucket)\n        batches.extend(bucket[i:i+batch_size] for i in range(0,len(bucket),batch_size))\n    rng.shuffle(batches)\n    return batches\n\ndef unique_windows(data,excluded=None):\n    seen=set(excluded or []); result=[]\n    for x,y in data:\n        identifier=hashlib.sha256(x.numpy().tobytes()+y.numpy().tobytes()).hexdigest()\n        if identifier in seen: continue\n        seen.add(identifier); result.append((x,y))\n    return result,seen\n\ndef loss(model, batch):\n    length = max(len(x) for x, _ in batch)\n    x = torch.full((len(batch), length), PAD, dtype=torch.long); y = torch.full_like(x, -100)\n    for i, (a, b) in enumerate(batch): x[i, :len(a)] = a; y[i, :len(b)] = b\n    device=next(model.parameters()).device\n    autocast=torch.autocast(\'cuda\',dtype=model._amp_dtype) if getattr(model,\'_amp_enabled\',False) else nullcontext()\n    with autocast:\n        return F.cross_entropy(model(x.to(device)).reshape(-1, 262), y.to(device).reshape(-1), ignore_index=-100)\n\ndef evaluate_loss(model, data, batch_size):\n    total = tokens = 0\n    for offset in range(0, len(data), batch_size):\n        batch = data[offset:offset+batch_size]\n        count = sum(int((y != -100).sum()) for _, y in batch)\n        total += float(loss(model, batch)) * count; tokens += count\n    return total / tokens if tokens else None\n\ndef train(args):\n    if args.steps < 1 or args.batch < 1 or args.threads < 1: raise ValueError(\'Positive steps, batch and threads required\')\n    if not math.isfinite(args.lr) or args.lr <= 0: raise ValueError(\'Learning rate must be finite and positive\')\n    continuous = getattr(args, \'continuous\', False)\n    save_every = getattr(args, \'save_every\', 100)\n    patience = getattr(args, \'patience\', 0)\n    if patience < 0: raise ValueError(\'Patience must be zero or positive\')\n    if save_every < 1: raise ValueError(\'Save interval must be positive\')\n    max_seconds=getattr(args,\'max_seconds\',0)\n    if not math.isfinite(max_seconds) or max_seconds<0: raise ValueError(\'Time limit must be zero or positive\')\n    torch.set_num_threads(args.threads); torch.manual_seed(args.seed); rng = random.Random(args.seed)\n    rows = read_jsonl(args.data)\n    snapshot = json.loads(Path(args.snapshot).read_text(encoding=\'utf-8\')) if args.snapshot else None\n    if any(row.get(\'source\', {}).get(\'kind\') == \'user\' for row in rows):\n        if continuous: raise ValueError(\'Continuous runs require seed/teacher data; user consent must be refreshed between bounded runs\')\n        if not snapshot or not 0 <= time.time() * 1000 - snapshot.get(\'createdAt\', 0) < 300000:\n            raise ValueError(\'User examples require a fresh consent check. Use scripts/model-data.mjs train.\')\n    # User records always require a fresh Firebase consent snapshot supplied by the wrapper.\n    rows = [row for row in rows if eligible(row, snapshot)]\n    if args.resume: model,resume_metadata=load(args.resume)\n    else: model,resume_metadata=Nebula(config_for_preset(args.preset)),{}\n    inherited_steps=resume_metadata.get(\'steps\',0)\n    requested_device=getattr(args,\'device\',\'cpu\')\n    device=\'cuda\' if requested_device==\'auto\' and torch.cuda.is_available() else \'cpu\' if requested_device==\'auto\' else requested_device\n    if device==\'cuda\' and not torch.cuda.is_available(): raise ValueError(\'CUDA GPU is unavailable in this PyTorch installation\')\n    model.to(device)\n    model._amp_enabled=device==\'cuda\' and getattr(args,\'amp\',False)\n    model._amp_dtype=torch.bfloat16 if device==\'cuda\' and torch.cuda.is_bf16_supported() else torch.float16\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=model._amp_enabled and model._amp_dtype==torch.float16)\n    objective=getattr(args,\'objective\',\'assistant\')\n    grouped=getattr(args,\'grouped_split\',False)\n    if grouped:\n        train_rows,validation_rows=partition_rows(rows,args.seed)\n        training=examples(train_rows,model.config.context,True,objective)\n        validation=examples(validation_rows,model.config.context,True,objective)\n        training,seen_windows=unique_windows(training)\n        validation,_=unique_windows(validation,seen_windows)\n        data=training+validation\n    else:\n        data = examples(rows, model.config.context, include_owner_rules=True,objective=objective)\n        rng.shuffle(data); split = max(1, int(len(data) * .9)); training, validation = data[:split], data[split:]\n    if not training: raise ValueError(\'No approved training examples fit the model context\')\n    if grouped and not validation: raise ValueError(\'No validation conversations fit the model context\')\n    available_validation=len(validation)\n    max_validation=getattr(args,\'max_validation\',0)\n    if max_validation:\n        validation_rng=random.Random(args.seed+1)\n        validation=validation_rng.sample(validation,min(len(validation),max_validation))\n    validation.sort(key=lambda pair:len(pair[0]))\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=.01)\n    with torch.no_grad(): initial = float(loss(model, training[:args.batch]))\n    if not math.isfinite(initial): initial=None\n    completed = 0\n    data_hash = hashlib.sha256(Path(args.data).read_bytes()).hexdigest()\n    best_path = Path(args.output).with_suffix(\'.best.pt\')\n    best_validation = float(\'inf\')\n    stale_checks = 0\n    batches=[]; epoch=0; completed_total_before=0\n    if best_path.exists():\n        _, previous = load(best_path)\n        if previous.get(\'dataSha256\') == data_hash and previous.get(\'seed\') == args.seed and previous.get(\'objective\',\'assistant\') == objective and previous.get(\'groupedSplit\',False) == grouped and previous.get(\'maxValidation\',0) == max_validation and previous.get(\'windowFormat\',1)==2:\n            previous_loss = previous.get(\'validationLoss\')\n            if isinstance(previous_loss, (int, float)): best_validation = previous_loss\n    state_enabled=getattr(args,\'save_training_state\',False)\n    if state_enabled and args.resume:\n        saved_state=torch.load(args.resume,map_location=\'cpu\',weights_only=True).get(\'trainingState\')\n        signature={\'dataSha256\':data_hash,\'objective\':objective,\'seed\':args.seed,\n                   \'batch\':args.batch,\'bucketed\':getattr(args,\'bucketed\',False),\n                   \'groupedSplit\':grouped,\'windowFormat\':2,\'maxValidation\':max_validation,\n                   \'amp\':model._amp_enabled,\'learningRate\':args.lr}\n        if saved_state and saved_state.get(\'signature\')==signature:\n            optimizer.load_state_dict(saved_state[\'optimizer\'])\n            scaler.load_state_dict(saved_state[\'scaler\'])\n            rng.setstate(saved_state[\'samplerRandom\'])\n            torch.set_rng_state(saved_state[\'torchRandom\'])\n            if device==\'cuda\' and saved_state.get(\'cudaRandom\'):\n                torch.cuda.set_rng_state_all(saved_state[\'cudaRandom\'])\n            batches=[[training[index] for index in batch] for batch in saved_state[\'remainingBatches\']]\n            epoch=saved_state[\'epoch\']; stale_checks=saved_state[\'staleChecks\']\n            completed_total_before=saved_state.get(\'stageSteps\',0)\n            print(json.dumps({\'trainingStateRestored\':True,\'previousStageSteps\':completed_total_before}),flush=True)\n        else:\n            print(json.dumps({\'trainingStateRestored\':False,\'reason\':\'legacy_checkpoint_or_new_training_configuration\'}),flush=True)\n    signature={\'dataSha256\':data_hash,\'objective\':objective,\'seed\':args.seed,\n               \'batch\':args.batch,\'bucketed\':getattr(args,\'bucketed\',False),\n               \'groupedSplit\':grouped,\'windowFormat\':2,\'maxValidation\':max_validation,\n               \'amp\':model._amp_enabled,\'learningRate\':args.lr}\n    training_indices={id(x):index for index,(x,_) in enumerate(training)}\n    def checkpoint(stopped=False,validate=True,reason=None):\n        nonlocal best_validation, stale_checks\n        model.eval()\n        current=validation_loss=None\n        if validate:\n            with torch.no_grad():\n                current = float(loss(model, training[:args.batch]))\n                validation_loss = evaluate_loss(model, validation, args.batch)\n            if not math.isfinite(current) or validation_loss is not None and not math.isfinite(validation_loss):\n                raise NumericalFailure(\'Non-finite checkpoint metric\')\n        report = {\'parameters\': sum(p.numel() for p in model.parameters()), \'steps\': inherited_steps+completed,\'stepsThisRun\':completed, \'trainingExamples\': len(training), \'validationExamples\': len(validation),\'availableValidationExamples\':available_validation,\'originalConversations\':len(rows), \'skippedExamples\': None if grouped else max(0,len(rows) - len(data)), \'initialLoss\': initial, \'finalLoss\': current, \'validationLoss\': validation_loss, \'seed\': args.seed, \'dataSha256\': data_hash, \'experimental\': True, \'qualityApproved\': False, \'stopped\': stopped,\'objective\':objective,\'groupedSplit\':grouped,\'maxValidation\':max_validation,\'device\':device,\'windowFormat\':2}\n        if reason: report[\'stopReason\']=reason\n        improved=validation_loss is not None and validation_loss < best_validation\n        if improved:\n            best_validation = validation_loss\n            stale_checks = 0\n        elif validation_loss is not None:\n            stale_checks += 1\n            if validation_loss > best_validation * 1.2:\n                print(\'Validation loss is worsening: this run is overfitting. More varied data is needed; Ctrl+C saves and stops.\', flush=True)\n        training_state=None\n        if state_enabled:\n            training_state={\'signature\':signature,\'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict(),\n                \'samplerRandom\':rng.getstate(),\'torchRandom\':torch.get_rng_state(),\n                \'cudaRandom\':torch.cuda.get_rng_state_all() if device==\'cuda\' else [],\n                \'remainingBatches\':[[training_indices[id(x)] for x,_ in batch] for batch in batches],\n                \'epoch\':epoch,\'staleChecks\':stale_checks,\'stageSteps\':completed_total_before+completed}\n            report[\'stageSteps\']=completed_total_before+completed\n            report[\'trainingStateIncluded\']=True\n        if state_enabled: save(model,args.output,report,training_state=training_state)\n        else: save(model,args.output,report)\n        if improved:\n            if state_enabled: save(model,best_path,report,training_state=training_state)\n            else: save(model,best_path,report)\n            print(json.dumps({\'bestCheckpoint\': str(best_path), \'validationLoss\': best_validation}), flush=True)\n        print(json.dumps({\'checkpoint\': str(args.output), **report}), flush=True)\n    # Save even before the first update, then periodically and on Ctrl+C.\n    started=time.monotonic(); trained_tokens=0\n    try:\n        # Resume already has a verified checkpoint and optimizer state. Avoid an\n        # extra validation check consuming patience before its first update.\n        if not (state_enabled and completed_total_before): checkpoint()\n        for step in itertools.count() if continuous else range(args.steps):\n            if getattr(args,\'bucketed\',False):\n                if not batches: batches=bucket_batches(training,args.batch,rng); epoch+=1\n                batch=batches.pop()\n            else: batch = [training[rng.randrange(len(training))] for _ in range(args.batch)]\n            model.train()\n            optimizer.zero_grad(set_to_none=True); value = loss(model, batch)\n            if not torch.isfinite(value): raise NumericalFailure(\'Non-finite training loss\')\n            scaler.scale(value).backward(); scaler.unscale_(optimizer)\n            gradient_norm=torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            if not torch.isfinite(gradient_norm): raise NumericalFailure(\'Non-finite gradients; optimizer update was blocked\')\n            scaler.step(optimizer); scaler.update()\n            completed = step + 1\n            trained_tokens+=sum(len(x) for x,_ in batch)\n            if step == 0 or completed % 25 == 0: print(json.dumps({\'step\': completed, \'loss\': round(float(value.detach()), 4),\'epoch\':epoch or None,\'tokensPerSecond\':round(trained_tokens/max(.001,time.monotonic()-started))}), flush=True)\n            if max_seconds and time.monotonic()-started>=max_seconds:\n                checkpoint(stopped=True,validate=False,reason=\'time_limit\')\n                return {\'steps\':completed,\'stopped\':True,\'reason\':\'time_limit\'}\n            if completed % save_every == 0:\n                checkpoint()\n                if patience and stale_checks >= patience:\n                    print(json.dumps({\'stopped\':\'validation_not_improving\',\'steps\':completed,\'bestCheckpoint\':str(best_path),\'note\':\'Latest weights and best validation weights are saved. Live website unchanged.\'}),flush=True)\n                    return {\'steps\':completed,\'stopped\':True,\'reason\':\'validation_not_improving\'}\n    except KeyboardInterrupt:\n        checkpoint(stopped=True,validate=False,reason=\'interrupted\')\n        print(\'Stopped. Latest weights saved; the live website was not changed.\', flush=True)\n        return {\'steps\': completed, \'stopped\': True}\n    except NumericalFailure:\n        optimizer.zero_grad(set_to_none=True)\n        try:\n            checkpoint(stopped=True,validate=False,reason=\'non_finite_loss_or_gradient\')\n            message=\'Stopped for a numerical error. Finite current weights saved without validation; the best checkpoint is preserved.\'\n        except ValueError:\n            message=\'Stopped for a numerical error. Invalid weights were not saved; existing checkpoints are preserved.\'\n        print(message+\' Resume from a healthy best checkpoint with a lower learning rate.\',flush=True)\n        return {\'steps\':completed,\'stopped\':True,\'reason\':\'non_finite_loss_or_gradient\'}\n    checkpoint()\n    report=load(args.output)[1]\n    return report\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=\'Train owned Nebula weights from scratch. CPU preset is experimental, not production quality.\')\n    parser.add_argument(\'--continuous\', action=\'store_true\'); parser.add_argument(\'--save-every\', type=int, default=100)\n    parser.add_argument(\'--save-training-state\',action=\'store_true\',help=\'Save and restore compatible optimizer, sampler and random states\')\n    parser.add_argument(\'--max-seconds\',type=float,default=0,help=\'Stop and save after this many seconds; 0 disables\')\n    parser.add_argument(\'--patience\', type=int, default=0, help=\'Stop after this many validation checks without improvement; 0 disables\')\n    parser.add_argument(\'--device\',choices=[\'auto\',\'cpu\',\'cuda\'],default=\'auto\')\n    parser.add_argument(\'--objective\',choices=[\'assistant\',\'language\'],default=\'assistant\')\n    parser.add_argument(\'--grouped-split\',action=\'store_true\'); parser.add_argument(\'--bucketed\',action=\'store_true\'); parser.add_argument(\'--max-validation\',type=int,default=0)\n    parser.add_argument(\'--data\', required=True); parser.add_argument(\'--output\', required=True); parser.add_argument(\'--snapshot\'); parser.add_argument(\'--resume\'); parser.add_argument(\'--preset\', choices=[\'cpu\', \'smoke\',\'cpu-medium\',\'cpu-large\',\'gpu\'], default=\'cpu\'); parser.add_argument(\'--steps\', type=int, default=500); parser.add_argument(\'--batch\', type=int, default=4); parser.add_argument(\'--threads\', type=int, default=2); parser.add_argument(\'--seed\', type=int, default=42); parser.add_argument(\'--lr\', type=float, default=.0005)\n    train(parser.parse_args())\n', 'model/rules.py': '"""Owner instructions shared by training and private inference."""\nimport json\nimport os\nfrom pathlib import Path\n\ndef owner_instructions():\n    configured = os.environ.get(\'NEBULA_RULES_FILE\')\n    path = Path(configured) if configured else Path(__file__).resolve().parent.parent / \'nebula-rules.json\'\n    value = json.loads(path.read_text(encoding=\'utf-8\'))\n    lines = value.get(\'instructions\')\n    if not isinstance(lines, list) or not lines or any(not isinstance(line, str) or not line.strip() for line in lines): raise ValueError(\'Invalid owner rules\')\n    extra = os.environ.get(\'NEBULA_OWNER_INSTRUCTIONS\', \'\')\n    text = \'\\n\'.join(lines + ([extra] if extra else []))\n    if len(text) > 16000: raise ValueError(\'Owner instructions too long\')\n    return text\n\ndef with_owner_rules(messages):\n    text = owner_instructions()\n    if messages and messages[0][\'role\'] == \'system\' and (messages[0][\'content\'] == text or messages[0][\'content\'].startswith(text + \'\\n\')): return messages\n    return [{\'role\': \'system\', \'content\': text}, *messages]\n', 'model/dataflow.py': '"""Reviewed datasets. These commands never call teachers with user chat content."""\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\nfrom urllib.request import Request, urlopen\nfrom urllib.parse import urlparse\nimport os\nimport time\nfrom teacher_http import request_json\nfrom rules import owner_instructions\n\ndef read_jsonl(path):\n    with Path(path).open(encoding=\'utf-8\') as stream:\n        return [json.loads(line) for line in stream if line.strip()]\n\ndef write_jsonl(path, rows):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(\'.tmp\')\n    temporary.write_text(\'\'.join(json.dumps(row, ensure_ascii=False) + \'\\n\' for row in rows), encoding=\'utf-8\')\n    temporary.replace(path)\n\ndef redact(text):\n    text = re.sub(r\'\\b[\\w.+-]+@[\\w.-]+\\.[a-zA-Z]{2,}\\b\', \'[email removed]\', text)\n    text = re.sub(r\'\\b(?:sk-|AIza|hf_|gsk_|AQ\\.)[\\w-]{15,}\\b\', \'[credential removed]\', text)\n    text = re.sub(r\'\\bBearer\\s+[\\w.~/+=-]{12,}\', \'[credential removed]\', text, flags=re.I)\n    text = re.sub(r\'(?<!\\w)(?:\\+?\\d[\\d ()-]{7,}\\d)(?!\\w)\', \'[number removed]\', text)\n    return text\n\ndef example_id(messages):\n    return hashlib.sha256(json.dumps(messages, sort_keys=True).encode()).hexdigest()\n\ndef eligible(row, snapshot=None):\n    if row.get(\'reviewStatus\') != \'approved\': return False\n    source = row.get(\'source\', {})\n    if source.get(\'kind\') == \'user\':\n        if not snapshot: return False\n        profile = snapshot.get(\'accounts\', {}).get(source.get(\'accountKey\'))\n        owner = snapshot.get(\'training\', {}).get(source.get(\'owner\'))\n        if not profile or profile.get(\'improve\') is not True or not owner or owner.get(\'revoked\'): return False\n        if profile.get(\'trainingGeneration\', 0) != source.get(\'generation\'): return False\n        original = owner.get(\'samples\', {}).get(source.get(\'chatId\'), {}).get(source.get(\'turnId\'))\n        if not original or original.get(\'consentVersion\') != 1: return False\n        # An export cannot be modified into different content while claiming old consent.\n        messages = [{\'role\': m[\'role\'], \'content\': redact(m[\'content\'])} for m in original.get(\'messages\', [])]\n        # Reviewers may correct the answer, but cannot replace the consented user prompt.\n        if not messages or not row.get(\'messages\') or row[\'messages\'][0] != messages[0]: return False\n    elif source.get(\'kind\') == \'teacher\':\n        if source.get(\'trainingAllowed\') is not True: return False\n    elif source.get(\'kind\') == \'public\':\n        allowed = (source.get(\'dataset\') == \'HuggingFaceTB/everyday-conversations-llama3.1-2k\' and source.get(\'subset\') == \'default\') or (source.get(\'dataset\') == \'HuggingFaceTB/smoltalk\' and source.get(\'subset\') == \'smol-magpie-ultra\') or (source.get(\'dataset\') == \'HuggingFaceTB/smol-smoltalk\' and source.get(\'subset\') == \'default\')\n        story = source.get(\'dataset\') == \'roneneldan/TinyStories\' and source.get(\'subset\') == \'default\' and source.get(\'license\') == \'cdla-sharing-1.0\'\n        educational = source.get(\'curriculum\') == \'v13-language\' and source.get(\'originalSplit\') == \'train\' and source.get(\'license\') == \'odc-by\' and ((source.get(\'dataset\') == \'HuggingFaceTB/smollm-corpus\' and source.get(\'subset\') == \'cosmopedia-v2\') or (source.get(\'dataset\') == \'HuggingFaceFW/fineweb-edu\' and source.get(\'subset\') == \'sample-10BT\'))\n        if not ((allowed and source.get(\'license\') == \'apache-2.0\') or story or educational) or not source.get(\'recordId\'): return False\n    elif source.get(\'kind\') != \'seed\': return False\n    messages = row.get(\'messages\')\n    # User exports remain a single consent-checked pair. Owned/teacher datasets may be dialogues.\n    maximum = 2 if source.get(\'kind\') == \'user\' else 12\n    return (isinstance(messages, list) and 2 <= len(messages) <= maximum and len(messages) % 2 == 0\n            and all(isinstance(m, dict) and m.get(\'role\') == (\'user\' if i % 2 == 0 else \'assistant\')\n                    and isinstance(m.get(\'content\'), str) and 0 < len(m[\'content\'].strip()) <= 16000\n                    for i, m in enumerate(messages)))\n\ndef prepare(inputs, output, snapshot=None):\n    rows, seen = [], set()\n    for path in inputs:\n        for row in read_jsonl(path):\n            if not eligible(row, snapshot): continue\n            identifier = example_id(row[\'messages\'])\n            if identifier in seen: continue\n            seen.add(identifier); rows.append({**row, \'id\': identifier})\n    if not rows: raise ValueError(\'No approved, currently eligible training examples\')\n    write_jsonl(output, rows)\n    return len(rows)\n\ndef generate_teacher(config, prompts, output, limit=10, transport=None):\n    """Bounded calls to an OpenAI-compatible API. Original prompts only."""\n    if not 1 <= limit <= 100: raise ValueError(\'Request limit must be 1 to 100\')\n    if config.get(\'trainingAllowed\') is not True: raise ValueError(\'Choose a teacher whose output may be used for training\')\n    endpoint = config[\'url\']; parsed = urlparse(endpoint)\n    if parsed.username or parsed.password or parsed.query: raise ValueError(\'Credentials belong in an environment variable\')\n    if parsed.scheme != \'https\' and not (parsed.scheme == \'http\' and parsed.hostname in [\'localhost\', \'127.0.0.1\', \'::1\']): raise ValueError(\'HTTPS required for remote teacher\')\n    key = os.environ.get(config.get(\'keyEnv\', \'NEBULA_TEACHER_KEY\'), \'\')\n    if not key and parsed.hostname not in [\'localhost\', \'127.0.0.1\', \'::1\']: raise ValueError(\'Teacher API key is missing\')\n    rows = read_jsonl(output) if Path(output).exists() else []\n    completed = {r.get(\'promptId\') for r in rows}\n    calls = 0\n    for prompt in prompts:\n        # Prevent accidental transmission of exported user conversations to teachers.\n        if prompt.get(\'source\') != \'original\' or not isinstance(prompt.get(\'prompt\'), str): raise ValueError(\'Teacher prompts must be original curriculum prompts\')\n        text = prompt[\'prompt\'].strip()\n        if not text or len(text) > 4000: raise ValueError(\'Invalid curriculum prompt\')\n        identifier = hashlib.sha256((config.get(\'name\', \'custom\') + \':\' + config[\'model\'] + \':\' + text).encode()).hexdigest()\n        if identifier in completed: continue\n        if calls >= limit: break\n        payload = {\'model\': config[\'model\'], \'messages\': [{\'role\': \'system\', \'content\': \'Create a training example for Nebula using these owner instructions.\\n\' + owner_instructions() + \'\\n\' + config.get(\'instructions\', \'\')}, {\'role\': \'user\', \'content\': text}], \'max_tokens\': min(int(config.get(\'maxTokens\', 256)), 1024), \'temperature\': 0.6}\n        if config.get(\'temperature\', 0.6) is None: payload.pop(\'temperature\')\n        if config.get(\'reasoningEffort\') in [\'low\',\'medium\',\'high\']: payload[\'reasoning_effort\'] = config[\'reasoningEffort\']\n        request = Request(endpoint, json.dumps(payload).encode(), {\'Content-Type\': \'application/json\', \'User-Agent\':\'Nebula-Teacher/1.0\', **({\'Authorization\': \'Bearer \' + key} if key else {})})\n        if calls and not transport:\n            delay = min(60, max(0, float(config.get(\'requestIntervalSeconds\', 0))))\n            if delay: time.sleep(delay)\n        if transport: data = transport(request)\n        else:\n            # Never follow a redirect carrying a credential to another host.\n            from urllib.request import build_opener, HTTPRedirectHandler\n            class NoRedirect(HTTPRedirectHandler):\n                def redirect_request(self, *args, **kwargs): return None\n            data = request_json(build_opener(NoRedirect), request, provider=config.get(\'name\',\'teacher\'), max_retries=int(config.get(\'httpRetries\',2)))\n        answer = data[\'choices\'][0][\'message\'][\'content\']\n        if not isinstance(answer, str) or not answer.strip() or len(answer) > 16000: raise ValueError(\'Invalid teacher answer\')\n        rows.append({\'promptId\': identifier, \'reviewStatus\': \'pending\', \'messages\': [{\'role\': \'user\', \'content\': text}, {\'role\': \'assistant\', \'content\': redact(answer)}], \'source\': {\'kind\': \'teacher\', \'provider\': config.get(\'name\', \'custom\'), \'model\': config[\'model\'], \'trainingAllowed\': True}})\n        calls += 1; completed.add(identifier); write_jsonl(output, rows)\n    return calls\n', 'model/teacher_http.py': '"""Bounded HTTP retries and credential-free quota diagnostics."""\nimport json\nimport math\nimport re\nimport time\nfrom urllib.error import HTTPError\n\nclass TeacherRequestError(ValueError):\n    def __init__(self, message, status=429, retry_after=60, retryable=True):\n        super().__init__(message)\n        self.status = status\n        self.retry_after = retry_after\n        self.retryable = retryable\n\ndef quota_info(error):\n    result = {\'zero\': False, \'daily\': False, \'retry\': None}\n    try:\n        raw = error.read(200001)\n        if len(raw) > 200000: return result\n        value = json.loads(raw).get(\'error\', {})\n        message = str(value.get(\'message\', \'\'))\n        result[\'zero\'] = bool(re.search(r\'limit\\s*:\\s*0(?:\\D|$)\', message, re.I))\n        for detail in value.get(\'details\', []):\n            if not isinstance(detail, dict): continue\n            if str(detail.get(\'@type\', \'\')).endswith(\'RetryInfo\'):\n                match = re.fullmatch(r\'(\\d+(?:\\.\\d+)?)s\', str(detail.get(\'retryDelay\', \'\')))\n                if match: result[\'retry\'] = math.ceil(float(match[1]))\n            if str(detail.get(\'@type\', \'\')).endswith(\'QuotaFailure\'):\n                for violation in detail.get(\'violations\', []):\n                    if not isinstance(violation, dict): continue\n                    if str(violation.get(\'quotaValue\', \'\')) == \'0\': result[\'zero\'] = True\n                    if \'perday\' in str(violation.get(\'quotaId\', \'\')).lower(): result[\'daily\'] = True\n        header = error.headers.get(\'Retry-After\') if error.headers else None\n        if header and header.isdigit(): result[\'retry\'] = max(result[\'retry\'] or 0, int(header))\n    except (ValueError, TypeError, AttributeError, OSError): pass\n    return result\n\ndef request_json(opener, request, sleep=time.sleep, provider=\'gemini\', max_retries=2):\n    label = \'Groq\' if provider == \'groq\' else \'Gemini secondary\' if provider == \'gemini-secondary\' else \'Gemini\' if provider == \'gemini\' else \'Teacher\'\n    quota_url = \'https://console.groq.com/settings/limits\' if provider == \'groq\' else \'https://aistudio.google.com/rate-limit\' if provider in [\'gemini\',\'gemini-secondary\'] else \'your provider quota dashboard\'\n    for attempt in range(max_retries + 1):\n        try:\n            with opener.open(request, timeout=60) as response:\n                raw = response.read(200001)\n                if len(raw) > 200000: raise ValueError(\'Teacher reply too large\')\n                return json.loads(raw)\n        except HTTPError as error:\n            if error.code != 429: raise\n            info = quota_info(error)\n            if info[\'zero\']:\n                raise TeacherRequestError(f\'{label} HTTP 429: this model has a zero quota limit for your project. Check {quota_url} and select a model with available quota. Retrying cannot change a zero limit.\', retryable=False) from None\n            if info[\'daily\']:\n                raise TeacherRequestError(f\'{label} HTTP 429: daily quota exhausted. Check {quota_url} for the reset. Completed examples remain saved.\', retryable=False) from None\n            delay = max(15 * (2 ** attempt), info[\'retry\'] or 0)\n            if attempt == max_retries or delay > 60:\n                hint = f\' Provider suggests waiting {info["retry"]} seconds.\' if info[\'retry\'] is not None else \'\'\n                raise TeacherRequestError(f\'{label} HTTP 429: rate or quota limit still blocks requests.\' + hint + f\' Check {quota_url}. Completed examples remain saved.\', retry_after=max(60,info[\'retry\'] or 0)) from None\n            print(f\'{label} rate limited. Waiting {delay} seconds before retry {attempt + 1}/2. Ctrl+C stops.\', flush=True)\n            sleep(delay)\n', 'model/public_data.py': '"""Download a bounded, cached public training sample; never call teacher APIs."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\nimport time\nfrom urllib.parse import urlencode\nfrom urllib.request import Request, urlopen\n\nSOURCES = {\n    \'everyday\': {\'dataset\':\'HuggingFaceTB/everyday-conversations-llama3.1-2k\',\'config\':\'default\',\'split\':\'train_sft\',\'license\':\'apache-2.0\',\'author\':\'Hugging Face\'},\n    \'magpie\': {\'dataset\':\'HuggingFaceTB/smoltalk\',\'config\':\'smol-magpie-ultra\',\'split\':\'train\',\'license\':\'apache-2.0\',\'author\':\'HuggingFaceTB\'},\n}\n\ndef fetch_page(source, offset, length, cache, transport=None):\n    path = cache / f\'{offset}-{length}.json\'\n    if path.exists(): return json.loads(path.read_text(encoding=\'utf-8\'))\n    query = urlencode({k:source[k] for k in [\'dataset\',\'config\',\'split\']} | {\'offset\':offset,\'length\':length})\n    url = \'https://datasets-server.huggingface.co/rows?\' + query\n    if transport: value = transport(url)\n    else:\n        for attempt in range(3):\n            try:\n                with urlopen(Request(url,headers={\'User-Agent\':\'Nebula-Public-Data/1.0\'}),timeout=45) as response:\n                    raw = response.read(12000001)\n                    if len(raw)>12000000: raise ValueError(\'Dataset page too large\')\n                    value = json.loads(raw)\n                break\n            except Exception:\n                if attempt == 2: raise\n                time.sleep(2 ** attempt)\n    if not isinstance(value.get(\'rows\'),list): raise ValueError(\'Invalid dataset page\')\n    cache.mkdir(parents=True,exist_ok=True)\n    temporary=path.with_suffix(\'.tmp\'); temporary.write_text(json.dumps(value),encoding=\'utf-8\'); temporary.replace(path)\n    return value\n\ndef convert(item, source, index):\n    messages = item.get(\'messages\')\n    if not isinstance(messages,list) or not 2<=len(messages)<=12 or len(messages)%2: return None\n    if any(not isinstance(m,dict) or m.get(\'role\')!=(\'user\' if i%2==0 else \'assistant\') or not isinstance(m.get(\'content\'),str) or not m[\'content\'].strip() or len(m[\'content\'])>16000 for i,m in enumerate(messages)): return None\n    # Identity boilerplate from a teacher is unsuitable as Nebula\'s identity training.\n    text=\' \'.join(m[\'content\'].lower() for m in messages if m[\'role\']==\'assistant\')\n    if any(phrase in text for phrase in [\'i am chatgpt\',\'i\\\'m chatgpt\',\'i am llama\',\'i\\\'m llama\',\'i am gemini\',\'i\\\'m gemini\']): return None\n    from dataflow import redact\n    messages=[{\'role\':m[\'role\'],\'content\':redact(m[\'content\'])} for m in messages]\n    identifier=hashlib.sha256(json.dumps(messages,sort_keys=True).encode()).hexdigest()\n    return {\'id\':identifier,\'reviewStatus\':\'approved\',\'approvalMethod\':\'public-dataset-structural-filter-unverified\',\'messages\':messages,\'source\':{\'kind\':\'public\',\'dataset\':source[\'dataset\'],\'subset\':source[\'config\'],\'license\':source[\'license\'],\'recordId\':identifier,\'rowIndex\':index,\'url\':\'https://huggingface.co/datasets/\'+source[\'dataset\']}}\n\ndef download(output, everyday=2260, magpie=5000, transport=None):\n    from dataflow import write_jsonl\n    if not 0<=everyday<=2260 or not 0<=magpie<=20000: raise ValueError(\'Bounded download requires 0-2260 everyday and 0-20000 magpie rows\')\n    output=Path(output); output.mkdir(parents=True,exist_ok=True)\n    rows=[]; seen=set(); counts={}\n    for name,limit in [(\'everyday\',everyday),(\'magpie\',magpie)]:\n        source=SOURCES[name]; offset=0; accepted=0\n        while offset<limit:\n            page=fetch_page(source,offset,min(100,limit-offset),output/\'public-cache\'/name,transport)\n            entries=page[\'rows\']\n            if not entries: break\n            for entry in entries:\n                # Dataset viewer may truncate large cells; never train their partial text.\n                if entry.get(\'truncated_cells\'): continue\n                row=convert(entry.get(\'row\',{}),source,entry.get(\'row_idx\',offset))\n                if row and row[\'id\'] not in seen: seen.add(row[\'id\']); rows.append(row); accepted+=1\n            offset+=len(entries)\n            print(json.dumps({\'publicDataset\':name,\'scanned\':offset,\'accepted\':accepted}),flush=True)\n            if offset>=page.get(\'num_rows_total\',limit): break\n        counts[name]=accepted\n    if not rows: raise ValueError(\'No valid public examples downloaded\')\n    write_jsonl(output/\'public-conversations.jsonl\',rows)\n    manifest={\'sources\':list(SOURCES.values()),\'counts\':counts,\'conversations\':len(rows),\'trainingDataSha256\':hashlib.sha256((output/\'public-conversations.jsonl\').read_bytes()).hexdigest(),\'changes\':[\'Removed malformed, duplicate and truncated records\',\'Redacted likely contact details and credentials\',\'Rejected selected foreign identity boilerplate\'],\'quality\':\'Not individually reviewed; public examples can contain mistakes.\'}\n    (output/\'public-sources.json\').write_text(json.dumps(manifest,indent=2),encoding=\'utf-8\')\n    print(json.dumps(manifest),flush=True)\n    return manifest\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\',default=str(Path(__file__).resolve().parent/\'data\'))\n    parser.add_argument(\'--everyday\',type=int,default=2260); parser.add_argument(\'--magpie\',type=int,default=5000)\n    args=parser.parse_args()\n    try: download(args.output,args.everyday,args.magpie)\n    except KeyboardInterrupt: sys.exit(\'Stopped; downloaded pages are cached. Rerun to continue.\')\n    except Exception as error: sys.exit(\'Public download failed (\'+type(error).__name__+\'). Cached pages are preserved; rerun to retry.\')\n', 'model/kaggle_migrate.py': '"""Restore verified V3 data and checkpoints into Kaggle\'s saved output folder."""\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\nimport zlib\n\nimport torch\nfrom nebula import load\n\nDATA_FILES={\n    \'model/data/stories-approved.jsonl\',\'model/data/improved-chat.jsonl\',\n    \'model/data/v3-language.jsonl\',\'model/data/v3-chat.jsonl\',\n    \'model/data/v3-foundations.jsonl\',\'model/data/v3-conversations.jsonl\',\n    \'model/data/stories-sources.json\',\'model/data/public-sources.json\',\n    \'model/DATA_SOURCES.md\',\'nebula-rules.json\',\n}\n\ndef sha256(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef restore_data(backup,project):\n    payload=torch.load(backup,map_location=\'cpu\',weights_only=True)\n    if payload.get(\'format\')!=\'nebula-v3-data-1\' or payload.get(\'compression\')!=\'zlib\':\n        raise ValueError(\'This is not a Nebula V3 data backup\')\n    entries=payload.get(\'files\',{}); hashes=payload.get(\'sha256\',{})\n    if set(entries)!=set(hashes) or not set(entries)<=DATA_FILES:\n        raise ValueError(\'Unexpected files in data backup\')\n    required={\'model/data/v3-chat.jsonl\',\'model/data/v3-language.jsonl\',\'nebula-rules.json\'}\n    if not required<=set(entries): raise ValueError(\'Required training data or rules are missing\')\n    decoded={}; total=0\n    for name,tensor in entries.items():\n        if not isinstance(tensor,torch.Tensor) or tensor.dtype!=torch.uint8 or tensor.ndim!=1:\n            raise ValueError(\'Invalid compressed data entry\')\n        decoder=zlib.decompressobj()\n        value=decoder.decompress(tensor.numpy().tobytes(),64*1024*1024+1)\n        total+=len(value)\n        if not decoder.eof or decoder.unused_data or total>256*1024*1024:\n            raise ValueError(\'Oversized or incomplete data backup\')\n        if hashlib.sha256(value).hexdigest()!=hashes[name]: raise ValueError(\'Training data checksum mismatch\')\n        decoded[name]=value\n    project=Path(project).resolve()\n    # Validate all existing targets before writing any file.\n    for name,value in decoded.items():\n        target=project/name\n        if not target.resolve().is_relative_to(project): raise ValueError(\'Unsafe destination\')\n        if target.exists() and sha256(target)!=hashes[name]:\n            raise ValueError(\'Existing training data differs; choose a new project folder\')\n    for name,value in decoded.items():\n        target=project/name; target.parent.mkdir(parents=True,exist_ok=True)\n        if target.exists(): continue\n        temporary=target.with_suffix(target.suffix+\'.restore-tmp\')\n        temporary.write_bytes(value); temporary.replace(target)\n    return hashes\n\ndef checkpoint_info(path,data_hash):\n    model,metadata=load(path); del model\n    if metadata.get(\'dataSha256\')!=data_hash or metadata.get(\'objective\')!=\'assistant\':\n        raise ValueError(\'Checkpoint does not match the backed-up V3 chat dataset\')\n    return metadata\n\ndef find_backup(input_root):\n    paths=sorted(Path(input_root).rglob(\'Nebula_V3_Data.pt\'))\n    if not paths: raise FileNotFoundError(\'Add your private dataset containing Nebula_V3_Data.pt and both v3-chat checkpoint files.\')\n    if len({sha256(p) for p in paths})!=1:\n        raise ValueError(\'Different V3 data backups are attached. Keep only the intended dataset version.\')\n    return paths[0]\n\ndef atomic_copy(source,target):\n    source=Path(source); target=Path(target)\n    if source.resolve()==target.resolve(): return\n    target.parent.mkdir(parents=True,exist_ok=True)\n    temporary=target.with_suffix(target.suffix+\'.copy-tmp\')\n    shutil.copyfile(source,temporary); temporary.replace(target)\n\ndef migrate(input_root,project):\n    input_root=Path(input_root); project=Path(project)\n    backup=find_backup(input_root)\n    hashes=restore_data(backup,project)\n    data_hash=hashes[\'model/data/v3-chat.jsonl\']\n    checkpoints=project/\'model/checkpoints\'\n    selected={}; summary={}\n    for filename in [\'v3-chat.pt\',\'v3-chat.best.pt\']:\n        candidates=list(input_root.rglob(filename))\n        target=checkpoints/filename\n        if target.exists(): candidates.append(target)\n        compatible=[]\n        for path in candidates:\n            info=checkpoint_info(path,data_hash)\n            compatible.append((path,info))\n        if not compatible: raise FileNotFoundError(\'Missing \'+filename+\'. Add both V3 model files as notebook inputs.\')\n        if filename.endswith(\'.best.pt\'):\n            def score(item):\n                value=item[1].get(\'validationLoss\')\n                return value if isinstance(value,(float,int)) and math.isfinite(value) else float(\'inf\')\n            source,info=min(compatible,key=score)\n        else: source,info=max(compatible,key=lambda item:item[1].get(\'steps\',0))\n        selected[target]=source\n        summary[filename]={\'savedSteps\':info.get(\'steps\'),\'validationLoss\':info.get(\'validationLoss\'),\n                           \'trainingStateIncluded\':info.get(\'trainingStateIncluded\',False)}\n    # Only install model files after both inputs have passed validation.\n    for target,source in selected.items(): atomic_copy(source,target)\n    atomic_copy(backup,project/\'Nebula_V3_Data.pt\')\n    summary[\'dataSha256\']=data_hash\n    summary[\'project\']=str(project)\n    summary[\'trainingStarted\']=False\n    (project/\'migration-report.json\').write_text(json.dumps(summary,indent=2),encoding=\'utf-8\')\n    print(json.dumps(summary,indent=2),flush=True)\n    return summary\n', 'model/research_client.py': '"""Private worker protocol. Tokens never go to object-storage URLs or logs."""\nimport base64\nimport hashlib\nimport os\nimport re\nimport shutil\nfrom pathlib import Path\nfrom urllib.parse import urlparse\nimport requests\n\nclass ResearchError(RuntimeError):\n    def __init__(self,code,status=0):super().__init__(str(code));self.code=code;self.status=status\n\ndef digests(path):\n    sha=hashlib.sha256();md5=hashlib.md5(usedforsecurity=False)\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda:stream.read(1024*1024),b\'\'):sha.update(chunk);md5.update(chunk)\n    return sha.hexdigest(),base64.b64encode(md5.digest()).decode()\n\ndef storage_url(url):\n    parsed=urlparse(url)\n    if parsed.scheme!=\'https\' or parsed.username or parsed.password or not (\n            parsed.hostname==\'storage.googleapis.com\' or (parsed.hostname or \'\').endswith(\'.storage.googleapis.com\')):\n        raise ResearchError(\'UNEXPECTED_STORAGE_HOST\')\n    return url\n\nclass Client:\n    def __init__(self,api=None,token=None):\n        self.api=(api or os.environ.get(\'NEBULA_RESEARCH_API\',\'\')).rstrip(\'/\')\n        self.token=token or os.environ.get(\'NEBULA_WORKER_TOKEN\',\'\')\n        parsed=urlparse(self.api)\n        if parsed.scheme!=\'https\' or parsed.username or parsed.password or parsed.query or parsed.fragment or len(self.token)<32:\n            raise ResearchError(\'WORKER_CONFIGURATION_MISSING\')\n    def call(self,action,body=None):\n        try:\n            response=requests.request(\'GET\' if body is None else \'POST\',self.api+\'/\'+action,\n                headers={\'Authorization\':\'Bearer \'+self.token},json=body,timeout=45,allow_redirects=False)\n            value=response.json()\n        except Exception:raise ResearchError(\'COORDINATOR_UNREACHABLE\') from None\n        if response.status_code!=200:raise ResearchError(value.get(\'code\',\'COORDINATOR_FAILED\'),response.status_code)\n        return value\n    def huggingface(self,ticket):\n        repo=os.environ.get(\'NEBULA_HF_REPO\',\'\')\n        token=os.environ.get(\'NEBULA_HF_TOKEN\',\'\')\n        if (not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',repo) or ticket.get(\'repo\')!=repo or\n                not re.fullmatch(r\'checkpoints/[a-f0-9-]{36}\\.pt\',ticket.get(\'path\',\'\')) or\n                not token.startswith(\'hf_\')):\n            raise ResearchError(\'HF_CONFIGURATION_MISSING\')\n        try:\n            from huggingface_hub import HfApi\n            hub=HfApi(endpoint=\'https://huggingface.co\',token=token)\n            if hub.model_info(repo).private is not True:raise ResearchError(\'HF_REPOSITORY_MUST_BE_PRIVATE\')\n            return hub,repo\n        except ResearchError:raise\n        except Exception:raise ResearchError(\'HF_ACCESS_OR_DEPENDENCY_FAILED\') from None\n    def download(self,identifier,destination):\n        ticket=self.call(\'download\',{\'id\':identifier})\n        if ticket.get(\'provider\')==\'huggingface\':return self.download_huggingface(ticket,destination)\n        url=storage_url(ticket[\'url\'])\n        path=Path(destination);path.parent.mkdir(parents=True,exist_ok=True)\n        if path.exists() and path.stat().st_size==ticket[\'bytes\'] and digests(path)[0]==ticket[\'sha256\']:return path\n        temporary=path.with_suffix(path.suffix+\'.download-tmp\');sha=hashlib.sha256();count=0\n        try:\n            with requests.get(url,stream=True,timeout=90,allow_redirects=False) as response:\n                if response.status_code!=200:raise ResearchError(\'CHECKPOINT_DOWNLOAD_FAILED\')\n                with temporary.open(\'wb\') as stream:\n                    for chunk in response.iter_content(1024*1024):\n                        count+=len(chunk)\n                        if count>ticket[\'bytes\'] or count>100*1024*1024:raise ResearchError(\'CHECKPOINT_SIZE_MISMATCH\')\n                        sha.update(chunk);stream.write(chunk)\n            if count!=ticket[\'bytes\'] or sha.hexdigest()!=ticket[\'sha256\']:raise ResearchError(\'CHECKPOINT_CHECKSUM_MISMATCH\')\n            temporary.replace(path);return path\n        except ResearchError:raise\n        except Exception:raise ResearchError(\'CHECKPOINT_DOWNLOAD_FAILED\') from None\n    def download_huggingface(self,ticket,destination):\n        hub,repo=self.huggingface(ticket)\n        if (not re.fullmatch(r\'[a-f0-9]{40}\',ticket.get(\'revision\',\'\')) or\n                not isinstance(ticket.get(\'bytes\'),int) or not 0<ticket[\'bytes\']<=100*1024*1024):\n            raise ResearchError(\'HF_INVALID_DOWNLOAD\')\n        path=Path(destination);path.parent.mkdir(parents=True,exist_ok=True)\n        if path.exists() and path.stat().st_size==ticket[\'bytes\'] and digests(path)[0]==ticket[\'sha256\']:return path\n        temporary=path.with_suffix(path.suffix+\'.download-tmp\')\n        try:\n            from huggingface_hub import hf_hub_download\n            cached=Path(hf_hub_download(repo_id=repo,filename=ticket[\'path\'],revision=ticket[\'revision\'],\n                                       token=os.environ[\'NEBULA_HF_TOKEN\'],endpoint=\'https://huggingface.co\'))\n            if cached.stat().st_size!=ticket[\'bytes\'] or digests(cached)[0]!=ticket[\'sha256\']:\n                raise ResearchError(\'CHECKPOINT_CHECKSUM_MISMATCH\')\n            shutil.copyfile(cached,temporary);temporary.replace(path);return path\n        except ResearchError:raise\n        except Exception:raise ResearchError(\'HF_DOWNLOAD_FAILED\') from None\n    def upload(self,path,kind=\'checkpoint\',parent=None,lease=None):\n        path=Path(path);sha,md5=digests(path)\n        ticket=self.call(\'upload\',{\'kind\':kind,\'bytes\':path.stat().st_size,\'sha256\':sha,\'md5\':md5,\n                                   \'parent\':parent,\'lease\':lease})\n        if ticket.get(\'provider\')==\'huggingface\':\n            hub,repo=self.huggingface(ticket)\n            try:\n                hub.upload_file(path_or_fileobj=str(path),path_in_repo=ticket[\'path\'],repo_id=repo,\n                                commit_message=\'Save private Nebula research checkpoint\')\n            except Exception:\n                try:self.call(\'finish\',{\'id\':ticket[\'id\']})\n                except Exception:raise ResearchError(\'HF_UPLOAD_FAILED\') from None\n            else:self.call(\'finish\',{\'id\':ticket[\'id\']})\n            return ticket[\'id\']\n        try:\n            with path.open(\'rb\') as stream:\n                response=requests.put(storage_url(ticket[\'url\']),data=stream,headers=ticket[\'headers\'],timeout=180,allow_redirects=False)\n                # A timeout or retry may occur after an upload succeeded. The\n                # coordinator checks the immutable object before accepting it.\n                if response.status_code not in [200,201,412]:raise ResearchError(\'CHECKPOINT_UPLOAD_FAILED\')\n        except Exception:\n            try:self.call(\'finish\',{\'id\':ticket[\'id\']})\n            except Exception:raise ResearchError(\'CHECKPOINT_UPLOAD_FAILED\') from None\n        else:self.call(\'finish\',{\'id\':ticket[\'id\']})\n        return ticket[\'id\']\n', 'model/research_train.py': '"""Synchronized one/two-GPU rounds; a single writer owns checkpoint files."""\nimport argparse\nfrom contextlib import nullcontext\nfrom datetime import timedelta\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport time\n\nimport torch\nimport torch.distributed as dist\nfrom torch.nn import functional as F\nfrom torch.nn.parallel import DistributedDataParallel\nfrom nebula import PAD,load,save\nfrom research_eval import research_rows,select_focus\nfrom train import examples,partition_rows,evaluate_loss,unique_windows\n\ndef global_batch(data,size,step,seed=42):\n    # Stateless, length-local sampling: every rank chooses the same global batch\n    # then consumes a different slice. No rank can run out of batches early.\n    rng=random.Random(seed+step);anchor=rng.randrange(len(data))\n    return [data[(anchor+i)%len(data)] for i in range(size)]\n\ndef summed_loss(model,batch,device,amp,dtype):\n    length=max(len(x) for x,_ in batch)\n    x=torch.full((len(batch),length),PAD,dtype=torch.long,device=device)\n    y=torch.full_like(x,-100)\n    for i,(a,b) in enumerate(batch):x[i,:len(a)]=a.to(device);y[i,:len(b)]=b.to(device)\n    with torch.autocast(\'cuda\',dtype=dtype) if amp else nullcontext():\n        value=F.cross_entropy(model(x).reshape(-1,262),y.reshape(-1),ignore_index=-100,reduction=\'sum\')\n    return value,int((y!=-100).sum())\n\ndef main(argv=None,on_progress=None):\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--data\',type=Path,required=True);parser.add_argument(\'--resume\',type=Path,required=True)\n    parser.add_argument(\'--output\',type=Path,required=True);parser.add_argument(\'--steps\',type=int,default=2000)\n    parser.add_argument(\'--batch\',type=int,default=16,help=\'GLOBAL batch shared by all GPUs\')\n    parser.add_argument(\'--device\',choices=[\'cpu\',\'cuda\'],default=\'cuda\')\n    parser.add_argument(\'--focus\',choices=[\'balanced\',\'identity\',\'memory\',\'math\',\'language\'],default=\'balanced\')\n    parser.add_argument(\'--lr\',type=float,default=.00005)\n    parser.add_argument(\'--max-seconds\',type=float,default=3600)\n    parser.add_argument(\'--benchmark\',action=\'store_true\')\n    parser.add_argument(\'--save-every\',type=int,default=200)\n    parser.add_argument(\'--sampling-seed\',type=int,default=42)\n    parser.add_argument(\'--best-checkpoint\',action=\'store_true\',help=\'Also preserve the lowest validation-loss checkpoint\')\n    parser.add_argument(\'--conversation-mix\',action=\'store_true\',help=\'80% conversation updates; 20% mixed, including memory\')\n    parser.add_argument(\'--review-data\',type=Path,help=\'Fresh consent-checked review feed, training only\')\n    parser.add_argument(\'--rendezvous\',help=\'File rendezvous for the local CPU distributed test\')\n    args=parser.parse_args(argv)\n    world=int(os.environ.get(\'WORLD_SIZE\',\'1\'));rank=int(os.environ.get(\'RANK\',\'0\'));local=int(os.environ.get(\'LOCAL_RANK\',\'0\'))\n    if rank==0 and not args.benchmark and on_progress is None and os.environ.get(\'NEBULA_JOB_LEASE\'):\n        from research_client import Client\n        client=Client();job=os.environ[\'NEBULA_JOB_LEASE\']\n        on_progress=lambda step:client.call(\'heartbeat\',{\'lease\':job})\n    if args.steps<1 or args.steps>10000 or args.save_every<1 or args.batch<world or args.batch%world or not math.isfinite(args.lr) or args.lr<=0 or not math.isfinite(args.max_seconds) or not 0<args.max_seconds<=7200:\n        parser.error(\'Use positive steps (up to 10000), LR, and a global batch divisible by worker count\')\n    if not 0<=args.sampling_seed<2**31:parser.error(\'Sampling seed must fit a nonnegative 31-bit integer\')\n    torch.set_num_threads(2);torch.manual_seed(args.sampling_seed)\n    if args.device==\'cuda\':torch.cuda.set_device(local);device=torch.device(\'cuda\',local)\n    else:device=torch.device(\'cpu\')\n    if world>1:\n        options={\'backend\':\'nccl\' if args.device==\'cuda\' else \'gloo\',\'timeout\':timedelta(minutes=5),\'rank\':rank,\'world_size\':world}\n        if args.rendezvous:options[\'init_method\']=Path(args.rendezvous).resolve().as_uri()\n        dist.init_process_group(**options)\n    def barrier():\n        if world>1:dist.barrier()\n    try:\n        model,metadata=load(args.resume);model.to(device)\n        amp=args.device==\'cuda\';dtype=torch.bfloat16 if amp and torch.cuda.is_bf16_supported() else torch.float16\n        model._amp_enabled=amp;model._amp_dtype=dtype\n        training_rows,validation_rows=partition_rows(research_rows(args.data),42)\n        training,seen=unique_windows(examples(training_rows,model.config.context,True,\'assistant\'))\n        validation,all_seen=unique_windows(examples(validation_rows,model.config.context,True,\'assistant\'),seen)\n        if not training or not validation:raise ValueError(\'Training and held-out validation data are required\')\n        focused,_=unique_windows(examples(select_focus(training_rows,args.focus),model.config.context,True,\'assistant\'))\n        chat_pool=[];mixed_pool=[];memory_pool=[];review_pool=[]\n        if args.conversation_mix:\n            from conversation_mix import conversational,mixed_lane\n            chat_rows=[r for r in training_rows if conversational(r)]\n            mixed_rows=[r for r in training_rows if not conversational(r)]\n            chat_pool,_=unique_windows(examples(chat_rows,model.config.context,True,\'assistant\'))\n            mixed_focus=select_focus(mixed_rows,args.focus) if args.focus not in [\'balanced\',\'language\'] else []\n            mixed_pool,_=unique_windows(examples(mixed_focus or mixed_rows,model.config.context,True,\'assistant\'))\n            memory_pool,_=unique_windows(examples(select_focus(mixed_rows,\'memory\'),model.config.context,True,\'assistant\'))\n            if not chat_pool or not mixed_pool or not memory_pool:raise ValueError(\'80/20 training requires conversation, mixed and memory examples\')\n            if args.review_data and args.review_data.is_file():\n                from review_feed import validated_rows\n                from dataflow import read_jsonl\n                rows=validated_rows(read_jsonl(args.review_data))\n                review_pool,_=unique_windows(examples(rows,model.config.context,True,\'assistant\'),all_seen)\n            for pool in [chat_pool,mixed_pool,memory_pool,review_pool]:pool.sort(key=lambda pair:len(pair[0]))\n            if rank==0:print(json.dumps({\'conversationPercent\':80,\'mixedPercent\':20,\'memoryIncluded\':True,\'reviewedTrainingWindows\':len(review_pool),\'mixMeasuredIn\':\'optimizer updates\'}),flush=True)\n        training.sort(key=lambda pair:len(pair[0]));focused.sort(key=lambda pair:len(pair[0]))\n        validation=random.Random(43).sample(validation,min(128,len(validation)))\n        validation.sort(key=lambda pair:len(pair[0]))\n        optimizer=torch.optim.AdamW(model.parameters(),lr=args.lr,weight_decay=.01)\n        scaler=torch.amp.GradScaler(\'cuda\',enabled=amp and dtype==torch.float16,init_scale=1024)\n        signature={\'format\':\'research-ddp-1\',\'dataSha256\':hashlib.sha256(args.data.read_bytes()).hexdigest(),\n                   \'world\':world,\'batch\':args.batch,\'focus\':args.focus,\'lr\':args.lr,\'amp\':str(dtype) if amp else \'none\',\'samplingSeed\':args.sampling_seed,\'conversationMix\':args.conversation_mix,\'reviewSha256\':hashlib.sha256(args.review_data.read_bytes()).hexdigest() if args.review_data and args.review_data.is_file() else None}\n        state=torch.load(args.resume,map_location=\'cpu\',weights_only=True).get(\'trainingState\',{})\n        if state.get(\'signature\')==signature:\n            optimizer.load_state_dict(state[\'optimizer\']);scaler.load_state_dict(state[\'scaler\'])\n        inherited=metadata.get(\'steps\',0)\n        wrapped=DistributedDataParallel(model,device_ids=[local] if amp else None,broadcast_buffers=False) if world>1 else model\n        mix_counts={\'conversationUpdates\':0,\'mixedUpdates\':0,\'memoryUpdates\':0,\'reviewUpdates\':0};completed=0;trained=0;started=time.monotonic();bad=0;best_value=float(\'inf\')\n        def checkpoint():\n            nonlocal best_value\n            barrier()\n            if rank==0 and not args.benchmark:\n                model.eval()\n                with torch.no_grad():value=evaluate_loss(model,validation,8)\n                if value is None or not math.isfinite(value):raise ValueError(\'Non-finite validation result\')\n                report={\'steps\':inherited+completed,\'stepsThisRun\':completed,\'validationLoss\':value,\n                        \'dataSha256\':signature[\'dataSha256\'],\'objective\':\'assistant\',\'experimental\':True,\n                        \'qualityApproved\':False,\'trainingStateIncluded\':True,\'worldSize\':world,\'focus\':args.focus,\'conversationPercent\':80 if args.conversation_mix else None,\'reviewedTrainingWindows\':len(review_pool),**mix_counts}\n                training_state={\'signature\':signature,\'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict()}\n                save(model,args.output,report,training_state=training_state)\n                if args.best_checkpoint and value<best_value:\n                    best_value=value\n                    save(model,args.output.with_suffix(\'.best.pt\'),report,training_state=training_state)\n                print(json.dumps({\'checkpoint\':str(args.output),**report}),flush=True)\n            barrier()\n        if args.best_checkpoint and not args.benchmark:checkpoint()\n        while completed<args.steps:\n            step=inherited+completed\n            pool=focused if focused and args.focus!=\'balanced\' and step%4==0 else training\n            if args.conversation_mix:\n                lane=mixed_lane(completed)\n                pool=chat_pool if lane==\'conversation\' else memory_pool if lane==\'memory\' else mixed_pool\n                # At most one in ten conversation batches uses checked real-chat corrections.\n                chat_number=4*(completed//5)+completed%5\n                if lane==\'conversation\' and review_pool and chat_number%10==9:pool=review_pool\n            batch=global_batch(pool,args.batch,step,args.sampling_seed)\n            portion=args.batch//world;batch=batch[rank*portion:(rank+1)*portion]\n            wrapped.train();optimizer.zero_grad(set_to_none=True)\n            value,count=summed_loss(wrapped,batch,device,amp,dtype)\n            counts=torch.tensor(count,device=device,dtype=torch.float64)\n            if world>1:dist.all_reduce(counts)\n            objective=value*world/counts\n            finite=torch.tensor(int(torch.isfinite(objective)),device=device)\n            if world>1:dist.all_reduce(finite,op=dist.ReduceOp.MIN)\n            if not finite.item():raise ValueError(\'Non-finite loss; round stopped\')\n            scaler.scale(objective).backward();scaler.unscale_(optimizer)\n            norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)\n            finite=torch.tensor(int(torch.isfinite(norm)),device=device)\n            if world>1:dist.all_reduce(finite,op=dist.ReduceOp.MIN)\n            if not finite.item():\n                optimizer.zero_grad(set_to_none=True);bad+=1\n                if not scaler.is_enabled() or bad>16:raise ValueError(\'Non-finite gradients; round stopped\')\n                scaler.update(new_scale=scaler.get_scale()/2);continue\n            scaler.step(optimizer);scaler.update();completed+=1;trained+=int(counts)\n            if args.conversation_mix:\n                mix_counts[\'conversationUpdates\' if lane==\'conversation\' else \'mixedUpdates\']+=1\n                if lane==\'memory\':mix_counts[\'memoryUpdates\']+=1\n                if pool is review_pool:mix_counts[\'reviewUpdates\']+=1\n            if rank==0 and completed%100==0:\n                if on_progress:on_progress(completed)\n                print(json.dumps({\'step\':completed,\'worldSize\':world,\'globalBatch\':args.batch,\n                                  \'targetsPerSecond\':round(trained/max(.001,time.monotonic()-started))}),flush=True)\n            if completed%args.save_every==0:checkpoint()\n            stop=torch.tensor(int(time.monotonic()-started>=args.max_seconds),device=device)\n            if world>1:dist.all_reduce(stop,op=dist.ReduceOp.MAX)\n            if stop.item():break\n        checkpoint()\n        if rank==0:\n            result={\'completedSteps\':completed,\'worldSize\':world,\'targetsPerSecond\':round(trained/max(.001,time.monotonic()-started)),\n                    \'peakGpuMemoryBytes\':torch.cuda.max_memory_allocated(device) if amp else 0}\n            args.output.with_suffix(\'.performance.json\').write_text(json.dumps(result),encoding=\'utf-8\')\n            print(json.dumps(result),flush=True)\n    except KeyboardInterrupt:\n        # Periodic checkpoints remain valid even if a worker is forcibly stopped.\n        if rank==0:print(\'Interrupted. Use the last complete checkpoint; other ranks will stop.\',flush=True)\n        raise\n    finally:\n        if world>1 and dist.is_initialized():dist.destroy_process_group()\n\nif __name__==\'__main__\':main()\n', 'model/research_eval.py': '"""Locked, limited diagnostics. These scores do not establish general chat quality."""\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\nimport torch\nfrom dataflow import eligible,read_jsonl\nfrom nebula import load\nfrom rules import with_owner_rules\nfrom train import examples,partition_rows,evaluate_loss,unique_windows\n\nROOT=Path(__file__).resolve().parent\nCASES=json.loads((ROOT/\'research_cases.json\').read_text(encoding=\'utf-8\'))\n\ndef research_rows(path):\n    rows=read_jsonl(path)\n    if any(row.get(\'source\',{}).get(\'kind\')==\'user\' for row in rows):\n        raise ValueError(\'Research workers accept public, owned and permitted teacher data only\')\n    questions={case.get(\'prompt\',\'\').casefold() for case in CASES[\'cases\']}\n    for case in CASES[\'cases\']:\n        questions.update(m[\'content\'].casefold() for m in case.get(\'messages\',[]) if m[\'role\']==\'user\')\n    names=[name.casefold() for name in CASES[\'reservedNames\']]\n    return [row for row in rows if eligible(row) and\n            not any(m[\'content\'].casefold() in questions or any(name in m[\'content\'].casefold() for name in names) for m in row[\'messages\'])]\n\ndef healthy(text):\n    if not text or \'\\ufffd\' in text: return False\n    words=re.findall(r"\\w+",text.lower())\n    if len(words)>12:\n        grams=[tuple(words[i:i+3]) for i in range(len(words)-2)]\n        if len(set(grams))<len(grams)*.65: return False\n    return True\n\ndef correct(case,answer):\n    if not healthy(answer): return False\n    if \'number\' in case:\n        # A correct intermediate number embedded in an incorrect final answer\n        # is not counted as success.\n        numbers=re.findall(r\'-?\\d+\',answer)\n        return bool(numbers) and numbers[-1]==str(case[\'number\'])\n    return bool(re.search(r\'\\b\'+re.escape(case[\'expected\'])+r\'\\b\',answer,re.I))\n\ndef evaluate(checkpoint,data,device=\'cuda\',output=None):\n    torch.set_num_threads(2)\n    model,metadata=load(checkpoint);model.to(device)\n    rows=research_rows(data);train_rows,valid_rows=partition_rows(rows,42)\n    _,seen=unique_windows(examples(train_rows,model.config.context,True,\'assistant\'))\n    validation,_=unique_windows(examples(valid_rows,model.config.context,True,\'assistant\'),seen)\n    import random\n    validation=random.Random(43).sample(validation,min(128,len(validation)))\n    validation.sort(key=lambda pair:len(pair[0]))\n    with torch.no_grad(): value=evaluate_loss(model,validation,8)\n    groups={name:[] for name in [\'identity\',\'math\',\'memory\',\'extraction\']};replies=[]\n    for case in CASES[\'cases\']:\n        messages=case.get(\'messages\') or [{\'role\':\'user\',\'content\':case[\'prompt\']}]\n        answer=model.reply(with_owner_rules(messages),max_tokens=96,temperature=0)\n        passed=correct(case,answer);groups[case[\'category\']].append(passed)\n        replies.append({\'category\':case[\'category\'],\'messages\':messages,\'reply\':answer,\'passed\':passed})\n    result={\'benchmark\':CASES[\'benchmark\'],\'loss\':value,\n            **{key:sum(scores)/len(scores) for key,scores in groups.items()},\n            \'health\':sum(healthy(r[\'reply\']) for r in replies)/len(replies)}\n    if output: Path(output).write_text(json.dumps({\'checkpoint\':str(checkpoint),\'metadata\':metadata,\'report\':result,\'replies\':replies},indent=2),encoding=\'utf-8\')\n    del model\n    if device==\'cuda\':torch.cuda.empty_cache()\n    return result\n\ndef focus_for(report):\n    if report[\'health\']<.8:return \'language\'\n    weakest=min([\'identity\',\'math\',\'memory\',\'extraction\'],key=lambda key:report[key])\n    return \'memory\' if weakest==\'extraction\' else weakest\n\ndef select_focus(rows,focus):\n    if focus==\'balanced\':return rows\n    if focus==\'language\':return [row for row in rows if row.get(\'source\',{}).get(\'kind\') in [\'public\',\'teacher\'] or row.get(\'source\',{}).get(\'curriculum\')==\'v5-conversation\']\n    def match(row):\n        answer=\' \'.join(m[\'content\'] for m in row[\'messages\'] if m[\'role\']==\'assistant\').lower()\n        if focus==\'identity\':return \'nebula\' in answer\n        if focus==\'math\':return bool(re.search(r\'\\d+\\s*[+\\-]\\s*\\d+\\s*=\',answer))\n        return row.get(\'splitGroup\',\'\').startswith((\'name-\',\'copy-\',\'color-\')) or len(row[\'messages\'])>2 or \'your name is\' in answer or \'bag is\' in answer\n    return [row for row in rows if match(row)]\n', 'model/research_worker.py': '"""Bounded trainer/evaluator workers coordinating through the private API."""\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport time\n\nimport torch\nfrom kaggle_migrate import migrate,restore_data\nfrom nebula import load\nfrom research_client import Client,ResearchError\nfrom research_eval import evaluate,focus_for\nfrom research_train import main as train_one\nfrom rules import owner_instructions\n\ndef configure_rules(project):\n    # Code can live above the trainer/evaluator data folders in combined mode.\n    # Bind both this process and its GPU subprocesses to the verified restored rules.\n    path=(Path(project)/\'nebula-rules.json\').resolve()\n    if not path.is_file():raise ResearchError(\'RESTORED_OWNER_RULES_MISSING\')\n    os.environ[\'NEBULA_RULES_FILE\']=str(path)\n    owner_instructions()\n\ndef prefer(base,next):\n    keys=[\'identity\',\'math\',\'memory\',\'extraction\',\'health\']\n    score=lambda value:sum(value[k] for k in keys[:4])/4\n    return all(next[k]>=base[k]-.05 for k in keys) and next[\'health\']>=base[\'health\'] and next[\'loss\']<=base[\'loss\']*1.10 and (\n        score(next)>=score(base)+.025 or score(next)>=score(base) and next[\'loss\']<base[\'loss\']*.995)\n\ndef training_plan(state):\n    rejected=max(0,min(3,int(state.get(\'rejectedRounds\') or 0)))\n    focus=\'balanced\'\n    report=(state.get(\'lastEvaluation\') or {}).get(\'report\') or {}\n    if rejected:\n        if report.get(\'health\',1)<.8:focus=\'language\'\n        else:\n            categories=sorted([\'identity\',\'math\',\'memory\',\'extraction\'],key=lambda key:report.get(key,0))\n            focus=categories[(rejected-1)%len(categories)]\n            if focus==\'extraction\':focus=\'memory\'\n    # Rejected checkpoints are never reused as the stable head. Change the\n    # optimization on retry instead of replaying the same deterministic run.\n    return {\'focus\':focus,\'lr\':.000025*(.5**rejected),\'rejectedRounds\':rejected}\n\ndef launch(project,resume,output,steps,gpus,focus,lease=None,benchmark=False,lr=.00005,max_seconds=3600,\n           data=None,batch=16,best=False,save_every=200,sampling_seed=42,conversation_mix=False,review_data=None,quality_anchor=None,teacher_data=None,large_model=False,microbatch=1,curriculum_v9=False,validation_data=None,curriculum_v10=False,expanded_public=False):\n    command=[sys.executable]\n    if gpus>1:command+=[\'-m\',\'torch.distributed.run\',\'--standalone\',\'--nproc_per_node\',str(gpus)]\n    command+=[str(Path(__file__).with_name(\'v10_train.py\' if curriculum_v10 else \'v9_train.py\' if curriculum_v9 else \'v8_train.py\' if large_model else \'v7_train.py\' if quality_anchor else \'research_train.py\')),\'--data\',str(data or project/\'model/data/v3-chat.jsonl\'),\n              \'--resume\',str(resume),\'--output\',str(output),\'--steps\',str(steps),\'--batch\',str(batch),\n              \'--focus\',focus,\'--lr\',str(lr),\'--max-seconds\',str(max_seconds),\'--save-every\',str(save_every),\'--sampling-seed\',str(sampling_seed)]\n    if expanded_public:\n        if not curriculum_v10:raise ValueError(\'Expanded public data requires the verified conversation trainer\')\n        command+=[\'--expanded-public\']\n    if validation_data:command+=[\'--validation-data\',str(validation_data)]\n    if large_model or curriculum_v9 or curriculum_v10:command+=[\'--microbatch\',str(microbatch)]\n    if quality_anchor:command+=[\'--quality-anchor\',str(quality_anchor)]\n    if teacher_data:command+=[\'--teacher-data\',str(teacher_data)]\n    if conversation_mix:command+=[\'--conversation-mix\']\n    if review_data:command+=[\'--review-data\',str(review_data)]\n    if best:command+=[\'--best-checkpoint\']\n    if benchmark:command+=[\'--benchmark\']\n    env=dict(os.environ)\n    if lease:env[\'NEBULA_JOB_LEASE\']=lease\n    else:env.pop(\'NEBULA_JOB_LEASE\',None)\n    process=subprocess.Popen(command,cwd=project,env=env,start_new_session=os.name!=\'nt\')\n    try:code=process.wait()\n    except BaseException:\n        if process.poll() is None:\n            if os.name!=\'nt\':os.killpg(process.pid,signal.SIGTERM)\n            else:process.terminate()\n            try:process.wait(timeout=10)\n            except subprocess.TimeoutExpired:\n                if os.name!=\'nt\':os.killpg(process.pid,signal.SIGKILL)\n                else:process.kill()\n                process.wait()\n        raise\n    if code:raise ResearchError(\'TRAINING_PROCESS_STOPPED\')\n    return json.loads(output.with_suffix(\'.performance.json\').read_text())\n\ndef choose_gpus(project,resume,mode):\n    count=min(2,torch.cuda.device_count())\n    if count<1:raise ResearchError(\'GPU_REQUIRED\')\n    if mode!=\'auto\':return min(count,int(mode))\n    if count==1:return 1\n    results={}\n    for devices in [1,2]:\n        try:results[devices]=launch(project,resume,project/f\'benchmark-{devices}.pt\',20,devices,\'balanced\',benchmark=True)[\'targetsPerSecond\']\n        except ResearchError:results[devices]=0\n    if not any(results.values()):raise ResearchError(\'GPU_BENCHMARK_FAILED\')\n    chosen=2 if results[2]>results[1]*1.05 else 1\n    print(json.dumps({\'gpuBenchmark\':results,\'selectedGPUs\':chosen}),flush=True)\n    return chosen\n\ndef bootstrap(client,project,input_root):\n    state=client.call(\'state\')\n    if state.get(\'head\'):return state\n    migrate(input_root,project)\n    checkpoint=project/\'model/checkpoints/v3-chat.best.pt\'\n    _,metadata=load(checkpoint)\n    data=client.upload(project/\'Nebula_V3_Data.pt\',\'dataset\')\n    model=client.upload(checkpoint)\n    return client.call(\'seed\',{\'dataset\':data,\'checkpoint\':model,\'dataSha256\':metadata[\'dataSha256\']})\n\ndef main(argv=None):\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--role\',choices=[\'trainer\',\'evaluator\'],required=True)\n    parser.add_argument(\'--project\',type=Path,required=True)\n    parser.add_argument(\'--input\',type=Path,default=Path(\'/kaggle/input\'))\n    parser.add_argument(\'--gpus\',choices=[\'auto\',\'1\',\'2\'],default=\'auto\')\n    parser.add_argument(\'--hours\',type=float,default=6)\n    parser.add_argument(\'--rounds\',type=int,default=12)\n    args=parser.parse_args(argv)\n    if not 0<args.hours<=8 or not 1<=args.rounds<=24:parser.error(\'Use up to 8 hours and 1-24 rounds\')\n    project=args.project.resolve();project.mkdir(parents=True,exist_ok=True)\n    if not torch.cuda.is_available():raise ResearchError(\'GPU_REQUIRED\')\n    torch.set_num_threads(2)\n    client=Client();deadline=time.monotonic()+args.hours*3600;rounds=0;gpus=None\n    if args.role==\'trainer\':bootstrap(client,project,args.input)\n    while rounds<args.rounds and time.monotonic()<deadline:\n        state=client.call(\'state\')\n        if not state.get(\'head\'):\n            print(\'Waiting for Kaggle to upload the starting model and data.\',flush=True);time.sleep(30);continue\n        backup=client.download(state[\'dataset\'],project/\'Nebula_V3_Data.pt\');restore_data(backup,project)\n        configure_rules(project)\n        if args.role==\'trainer\' and gpus is None:\n            head=client.download(state[\'head\'],project/\'benchmark-source.pt\')\n            gpus=choose_gpus(project,head,args.gpus)\n        try:claimed=client.call(\'claim\',{})\n        except ResearchError as error:\n            if error.code in [\'WORKER_BUSY\',\'WAITING_FOR_OTHER_WORKER\']:\n                print(json.dumps({\'waiting\':error.code,\'role\':args.role}),flush=True);time.sleep(30);continue\n            raise\n        job=claimed[\'lease\'];lease=job[\'id\']\n        if job[\'role\']!=args.role:raise ResearchError(\'WRONG_WORKER_SECRET\')\n        base=client.download(job[\'base\'],project/\'base.pt\')\n        if args.role==\'trainer\':\n            output=project/f\'round-{claimed["revision"]}.pt\'\n            plan=training_plan(claimed)\n            print(json.dumps({\'trainingPlan\':plan,\'base\':job[\'base\']}),flush=True)\n            launch(project,base,output,2000,gpus,plan[\'focus\'],lease,lr=plan[\'lr\'],max_seconds=max(1,min(3600,deadline-time.monotonic())))\n            client.call(\'heartbeat\',{\'lease\':lease})\n            candidate=client.upload(output,parent=job[\'base\'],lease=lease)\n            client.call(\'submit\',{\'lease\':lease,\'id\':candidate})\n            print(json.dumps({\'submittedForEvaluation\':candidate,\'base\':job[\'base\']}),flush=True)\n        else:\n            candidate=client.download(job[\'candidate\'],project/\'candidate.pt\')\n            data=project/\'model/data/v3-chat.jsonl\'\n            client.call(\'heartbeat\',{\'lease\':lease})\n            baseline=evaluate(base,data,output=project/\'baseline-evaluation.json\')\n            client.call(\'heartbeat\',{\'lease\':lease})\n            trained=evaluate(candidate,data,output=project/\'candidate-evaluation.json\')\n            focus=focus_for(trained);experiment=project/\'experiment.pt\'\n            print(json.dumps({\'baseline\':baseline,\'trained\':trained,\'experimentFocus\':focus}),flush=True)\n            launch(project,candidate,experiment,200,1,focus,lease,lr=.000025,max_seconds=max(1,min(3600,deadline-time.monotonic())))\n            client.call(\'heartbeat\',{\'lease\':lease})\n            adapted=evaluate(experiment,data,output=project/\'experiment-evaluation.json\')\n            decision={\'lease\':lease,\'candidate\':job[\'candidate\'],\'baseline\':baseline,\'trained\':trained}\n            # Only upload a bounded experiment if it can beat both alternatives.\n            selected=trained if prefer(baseline,trained) else baseline\n            if prefer(baseline,adapted) and prefer(selected,adapted):\n                identifier=client.upload(experiment,parent=job[\'candidate\'],lease=lease)\n                decision[\'experiment\']={\'id\':identifier,\'report\':adapted}\n            final=client.call(\'decide\',decision)\n            print(json.dumps({\'evaluationDecision\':final[\'lastEvaluation\'],\'publicWebsiteUpdated\':False}),flush=True)\n        rounds+=1\n        (project/\'worker-state.json\').write_text(json.dumps({\'completedRounds\':rounds,\'role\':args.role}),encoding=\'utf-8\')\n    print(\'Worker budget finished. Checkpoints remain in object storage and local notebook outputs.\',flush=True)\n\nif __name__==\'__main__\':\n    try:main()\n    except ResearchError as error:print(json.dumps({\'workerStopped\':str(error.code),\'status\':error.status}),flush=True);sys.exit(1)\n', 'model/research_cases.json': '{\n  "benchmark": "nebula-research-v1",\n  "reservedNames": ["Maribel", "Keon", "Ayumi"],\n  "cases": [\n    {"category":"identity","prompt":"Which AI assistant is replying to me?","expected":"Nebula"},\n    {"category":"identity","prompt":"Please identify the assistant helping me.","expected":"Nebula"},\n    {"category":"math","prompt":"Compute 37 + 48.","number":85},\n    {"category":"math","prompt":"Compute 46 + 17.","number":63},\n    {"category":"math","prompt":"Compute 58 + 29.","number":87},\n    {"category":"math","prompt":"Compute 64 + 27.","number":91},\n    {"category":"memory","prompt":"Please call me Maribel. Repeat my name.","expected":"Maribel"},\n    {"category":"memory","prompt":"I introduced myself as Keon. Who am I?","expected":"Keon"},\n    {"category":"memory","messages":[{"role":"user","content":"Please call me Ayumi."},{"role":"assistant","content":"Nice to meet you, Ayumi."},{"role":"user","content":"Which name did I introduce myself with?"}],"expected":"Ayumi"},\n    {"category":"extraction","prompt":"The ribbon is teal. State the color of the ribbon.","expected":"teal"},\n    {"category":"extraction","prompt":"The box contains a spoon. What is inside the box?","expected":"spoon"}\n  ]\n}\n', 'model/v4_data.py': '"""Broader owned exercises, bounded public additions, and frozen diagnostics."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\nimport re\nfrom dataflow import eligible,example_id,read_jsonl,write_jsonl\nfrom nebula import encode\nfrom public_data import SOURCES,convert,fetch_page\nfrom research_eval import CASES\nfrom rules import with_owner_rules\n\nVERSION=\'nebula-v4-1\'\ndef digest(text):return hashlib.sha256(text.encode()).hexdigest()\ndef held(group):return int(digest(VERSION+\':\'+group)[:8],16)%10==0\nRESERVED={name.casefold() for name in CASES[\'reservedNames\']}\nLOCKED_PAIRS={(37,48),(46,17),(58,29),(64,27)}\n\ndef exercises():\n    rows=[];checks=[];seen=set()\n    def add(q,a,group,category,expected=None,number=None,messages=None):\n        messages=messages or [{\'role\':\'user\',\'content\':q},{\'role\':\'assistant\',\'content\':a}]\n        identifier=example_id(messages)\n        if identifier in seen:return\n        seen.add(identifier)\n        if held(group):\n            case={\'category\':category,\'messages\':messages[:-1],\'group\':group}\n            if expected is not None:case[\'expected\']=expected\n            if number is not None:case[\'number\']=number\n            checks.append(case)\n        else:\n            rows.append({\'id\':identifier,\'reviewStatus\':\'approved\',\'splitGroup\':group,\n                         \'source\':{\'kind\':\'seed\',\'curriculum\':VERSION},\'messages\':messages})\n    identities=[\'Who are you?\',\'What is your name?\',\'Introduce yourself.\',\n        \'Which assistant am I chatting with?\',\'What should I call this assistant?\',\n        \'Tell me the name of this AI.\',\'Can you tell me who you are?\',\n        \'Name the assistant in this conversation.\',\'How do you introduce yourself?\',\n        \'What name do you use?\',\'Who is answering my questions?\',\n        \'Tell me your identity.\',\'What is the assistant called?\',\n        \'Please give your name.\',\'Who am I speaking to?\',\'What do people call you?\',\n        \'Say the name you go by.\',\'Identify yourself briefly.\',\n        \'Can you introduce the assistant?\',\'Tell me which AI this is.\']\n    for i,question in enumerate(identities):\n        for prefix in [\'\', \'Hello. \', \'Hi there! \', \'Before we begin, \', \'Quick question: \']:\n            for suffix in [\'\', \' Keep it short.\', \' Please answer briefly.\', \' One sentence is enough.\']:\n                add(prefix+question+suffix,\'I am Nebula AI.\',\'identity-\'+str(i),\'identity\',expected=\'Nebula\')\n    rng=random.Random(20261017)\n    syllables=[\'ba\',\'be\',\'bi\',\'bo\',\'bu\',\'ca\',\'ce\',\'ci\',\'co\',\'cu\',\'da\',\'de\',\'di\',\'do\',\'du\',\n               \'fa\',\'fe\',\'fi\',\'fo\',\'fu\',\'ga\',\'ge\',\'gi\',\'go\',\'gu\',\'ha\',\'he\',\'hi\',\'ho\',\'hu\',\n               \'ja\',\'je\',\'ji\',\'jo\',\'ju\',\'ka\',\'ke\',\'ki\',\'ko\',\'ku\',\'la\',\'le\',\'li\',\'lo\',\'lu\',\n               \'ma\',\'me\',\'mi\',\'mo\',\'mu\',\'na\',\'ne\',\'ni\',\'no\',\'nu\',\'pa\',\'pe\',\'pi\',\'po\',\'pu\',\n               \'ra\',\'re\',\'ri\',\'ro\',\'ru\',\'sa\',\'se\',\'si\',\'so\',\'su\',\'ta\',\'te\',\'ti\',\'to\',\'tu\',\n               \'va\',\'ve\',\'vi\',\'vo\',\'vu\',\'za\',\'ze\',\'zi\',\'zo\',\'zu\']\n    names=set()\n    while len(names)<1200:\n        name=\'\'.join(rng.choice(syllables) for _ in range(rng.randint(2,5))).capitalize()\n        if not any(reserved in name.casefold() for reserved in RESERVED):names.add(name)\n    for name in sorted(names):\n        group=\'name-\'+name\n        for q in [f\'My name is {name}. What is my name?\',f\'Call me {name}. Repeat my name.\',\n                  f\'I am {name}. Say my name back to me.\',f\'Remember this name: {name}. Which name did I say?\',\n                  f\'Please address me as {name}. How should you address me?\']:\n            add(q,f\'Your name is {name}.\',group,\'memory\',expected=name)\n        messages=[{\'role\':\'user\',\'content\':f\'My name is {name}.\'},\n                  {\'role\':\'assistant\',\'content\':f\'Hello, {name}.\'},\n                  {\'role\':\'user\',\'content\':\'What name did I give you?\'},\n                  {\'role\':\'assistant\',\'content\':f\'Your name is {name}.\'}]\n        add(None,None,group,\'memory\',expected=name,messages=messages)\n    # Whole operand pairs are held out, including every wording for those pairs.\n    pairs=[(a,b) for a in range(100) for b in range(100) if (a,b) not in LOCKED_PAIRS and (b,a) not in LOCKED_PAIRS]\n    rng.shuffle(pairs)\n    for a,b in pairs[:5000]:\n        group=f\'addition-{min(a,b)}-{max(a,b)}\'\n        for q in [f\'What is {a} + {b}?\',f\'Add {a} and {b}.\',f\'Calculate the sum of {a} and {b}.\']:\n            add(q,f\'{a} + {b} = {a+b}.\',group,\'math\',number=a+b)\n    colors=[\'red\',\'green\',\'blue\',\'yellow\',\'purple\',\'orange\',\'pink\',\'white\',\'black\',\'gray\',\n            \'silver\',\'gold\',\'brown\',\'cyan\',\'magenta\',\'violet\',\'indigo\',\'navy\',\'maroon\',\'lime\']\n    objects=[\'kite\',\'cup\',\'hat\',\'shoe\',\'chair\',\'table\',\'button\',\'basket\',\'bottle\',\'book\',\n             \'jacket\',\'flower\',\'car\',\'train\',\'door\',\'gate\',\'plate\',\'mug\',\'pencil\',\'lamp\',\n             \'scarf\',\'sock\',\'coat\',\'bicycle\',\'boat\',\'balloon\',\'flag\',\'bead\',\'stone\',\'shell\']\n    for item in objects:\n        for color in colors:\n            group=f\'color-{item}-{color}\'\n            for q in [f\'The {item} is {color}. What color is it?\',\n                      f\'I have a {color} {item}. Tell me its color.\',\n                      f\'Look at the {item}: it is {color}. Repeat its color.\']:\n                add(q,f\'The {item} is {color}.\',group,\'extraction\',expected=color)\n    # Also practice exact unfamiliar strings, rather than a fixed vocabulary of names.\n    for index in range(800):\n        word=\'\'.join(rng.choice(\'abcdefghijklmnopqrstuvwxyz\') for _ in range(rng.randint(2,12)))\n        if any(name in word for name in RESERVED):continue\n        add(f\'Repeat this word exactly: {word}\',word,\'copy-\'+word,\'memory\',expected=word)\n    for i,(q,a) in enumerate([\n        (\'Hi\',\'Hello! How can I help you?\'),(\'Hello\',\'Hello! How can I help you?\'),\n        (\'Can you help me write an email?\',\'Yes. Who is it for, and what would you like to say?\'),\n        (\'I need help.\',\'What would you like help with?\'),\n        (\'What if you do not know?\',\'I can say I am unsure and ask for more information.\'),\n        (\'Do you make mistakes?\',\'Yes. Please check important answers.\'),\n        (\'Thanks!\',\'You are welcome!\')]):\n        for prefix in [\'\', \'Hey, \', \'Hi there. \']:\n            add(prefix+q,a,\'small-chat-\'+str(i),\'chat\',expected=None)\n    # Check cases without exact validators are qualitative only, not used for selection.\n    checks=[case for case in checks if \'expected\' in case or \'number\' in case]\n    return rows,checks\n\ndef benchmark(checks):\n    rng=random.Random(49317);result=[]\n    for category,limit in [(\'identity\',8),(\'math\',20),(\'memory\',16),(\'extraction\',12)]:\n        pool=[case for case in checks if case[\'category\']==category]\n        # Use one prompt per held-out scenario in the scored benchmark.\n        by_group={case[\'group\']:case for case in pool}\n        # Identity has few distinct phrase families; allow several variants there.\n        pool=list(by_group.values()) if category!=\'identity\' else pool\n        result.extend(rng.sample(pool,min(limit,len(pool))))\n    return {\'benchmark\':VERSION,\'cases\':CASES[\'cases\']+result,\'legacyCaseCount\':len(CASES[\'cases\'])}\n\ndef blocked_questions(checks):\n    blocked_questions={m[\'content\'].casefold() for case in CASES[\'cases\'] for m in case.get(\'messages\',[{\'role\':\'user\',\'content\':case.get(\'prompt\',\'\')}]) if m[\'role\']==\'user\'}\n    blocked_questions.update(m[\'content\'].casefold() for case in checks for m in case.get(\'messages\',[]) if m[\'role\']==\'user\')\n    return blocked_questions\n\ndef forbidden(row,checks,blocked=None):\n    text=\' \'.join(m[\'content\'].casefold() for m in row[\'messages\'])\n    if any(name in text for name in RESERVED):return True\n    blocked=blocked_questions(checks) if blocked is None else blocked\n    if any(m[\'content\'].casefold() in blocked for m in row[\'messages\'] if m[\'role\']==\'user\'):return True\n    # Avoid legacy public/teacher paraphrases of held arithmetic/name facts.\n    for message in row[\'messages\'][::2]:\n        nums=re.findall(r\'\\b\\d+\\b\',message[\'content\'])\n        if len(nums)==2:\n            a,b=map(int,nums);group=f\'addition-{min(a,b)}-{max(a,b)}\'\n            if (a,b) in LOCKED_PAIRS or (b,a) in LOCKED_PAIRS or held(group):return True\n    return False\n\ndef public_additions(directory,limit=1000,transport=None):\n    directory=Path(directory);source=SOURCES[\'magpie\'];rows=[]\n    for offset in range(5000,5000+limit,100):\n        try:page=fetch_page(source,offset,min(100,5000+limit-offset),directory/\'cache\',transport)\n        except Exception:\n            print(json.dumps({\'publicDownloadDeferred\':True,\'cachedAndOwnedDataStillAvailable\':True}),flush=True);break\n        for entry in page[\'rows\']:\n            if entry.get(\'truncated_cells\'):continue\n            row=convert(entry.get(\'row\',{}),source,entry.get(\'row_idx\',offset))\n            if row:rows.append(row)\n        print(json.dumps({\'publicRowsScannedThrough\':offset+len(page[\'rows\'])}),flush=True)\n    return rows\n\ndef prepare(project,extra_public=None):\n    project=Path(project);directory=project/\'model/data\';directory.mkdir(parents=True,exist_ok=True)\n    own,held_cases=exercises();cases=benchmark(held_cases)\n    checks_path=directory/\'v4-checks.json\';checks_path.write_text(json.dumps(cases,indent=2),encoding=\'utf-8\')\n    rows=[];seen=set();counts={};max_bytes=0;blocked=blocked_questions(held_cases)\n    def keep(row):\n        nonlocal max_bytes\n        if not eligible(row) or forbidden(row,held_cases,blocked):return\n        size=len(encode(with_owner_rules(row[\'messages\']))[0])\n        if size>1024:return\n        identifier=example_id(row[\'messages\'])\n        if identifier in seen:return\n        seen.add(identifier);rows.append({**row,\'id\':identifier});max_bytes=max(max_bytes,size)\n        category=row.get(\'source\',{}).get(\'curriculum\',row[\'source\'][\'kind\'])\n        counts[category]=counts.get(category,0)+1\n    for row in own:keep(row)\n    legacy=directory/\'v3-chat.jsonl\'\n    for row in (read_jsonl(legacy) if legacy.exists() else [])+(extra_public or []):\n        # Rehearse complete short public/teacher turns; do not copy raw user chats.\n        if row.get(\'source\',{}).get(\'kind\') not in [\'public\',\'teacher\']:continue\n        for i in range(0,len(row[\'messages\'])-1,2):\n            pair=row[\'messages\'][i:i+2]\n            if len(pair[0][\'content\'].encode())>240 or len(pair[1][\'content\'].encode())>400:continue\n            value={**row,\'messages\':pair,\'splitGroup\':\'source-\'+str(row[\'source\'].get(\'recordId\',row.get(\'id\',\'\')))}\n            keep(value)\n    data=directory/\'v4-chat.jsonl\';write_jsonl(data,rows)\n    manifest={\'version\':VERSION,\'conversations\':len(rows),\'counts\':counts,\'maximumContextBytes\':max_bytes,\n              \'dataSha256\':hashlib.sha256(data.read_bytes()).hexdigest(),\'diagnosticCases\':len(cases[\'cases\']),\n              \'checksSha256\':hashlib.sha256(checks_path.read_bytes()).hexdigest(),\n              \'sources\':[{\'dataset\':\'HuggingFaceTB/smoltalk\',\'subset\':\'smol-magpie-ultra\',\'license\':\'apache-2.0\',\'url\':\'https://huggingface.co/datasets/HuggingFaceTB/smoltalk\'},\n                         {\'dataset\':\'HuggingFaceTB/everyday-conversations-llama3.1-2k\',\'license\':\'apache-2.0\'},\n                         {\'kind\':\'original verified exercises\',\'version\':VERSION}],\n              \'note\':\'Public/teacher replies are structurally filtered, not independently verified. Owned arithmetic and copying targets are generated exactly. Legacy weights cannot be decontaminated retroactively.\'}\n    (directory/\'v4-manifest.json\').write_text(json.dumps(manifest,indent=2),encoding=\'utf-8\')\n    print(json.dumps({\'v4Data\':manifest}),flush=True)\n    return data,checks_path,manifest\n', 'model/v4_grow.py': '"""Increase owned model depth while initially preserving its forward function."""\nfrom dataclasses import asdict\nimport torch\nfrom nebula import Config,Nebula,load,save\n\ndef grow_model(model,layers):\n    if not model.config.layers<=layers<=model.config.layers+8:raise ValueError(\'Growth must add at most eight layers\')\n    config=asdict(model.config);config[\'layers\']=layers\n    with torch.random.fork_rng(devices=[]):\n        torch.manual_seed(49317);grown=Nebula(Config(**config))\n    missing,unexpected=grown.load_state_dict(model.state_dict(),strict=False)\n    if unexpected or any(not name.startswith(\'blocks.\') or int(name.split(\'.\')[1])<model.config.layers for name in missing):\n        raise ValueError(\'Unexpected architecture mismatch\')\n    with torch.no_grad():\n        for block in grown.blocks[model.config.layers:]:\n            block.self_attn.out_proj.weight.zero_();block.self_attn.out_proj.bias.zero_()\n            block.linear2.weight.zero_();block.linear2.bias.zero_()\n    grown.eval();model.eval()\n    # New residual blocks initially add zero; this must hold numerically.\n    generator=torch.Generator().manual_seed(49317)\n    probe=torch.randint(0,256,(1,min(64,model.config.context)),generator=generator)\n    with torch.no_grad():\n        torch.testing.assert_close(grown(probe),model.cpu()(probe),rtol=0,atol=0)\n    return grown\n\ndef grow_checkpoint(source,destination,layers):\n    original,metadata=load(source);grown=grow_model(original,layers)\n    info={**metadata,\'growth\':{\'originalLayers\':original.config.layers,\'layers\':layers,\n          \'parameters\':sum(p.numel() for p in grown.parameters()),\'initialFunctionPreserved\':True},\n          \'trainingStateIncluded\':False,\'qualityApproved\':False,\'experimental\':True}\n    save(grown,destination,info)\n    return info\n', 'model/v4_eval.py': '"""Frozen task checks plus an independent, grouped validation loss."""\nimport json\nimport random\nfrom pathlib import Path\nimport torch\nfrom dataflow import read_jsonl\nfrom nebula import load\nfrom research_eval import correct,healthy\nfrom rules import with_owner_rules\nfrom train import examples,partition_rows,unique_windows,evaluate_loss\n\nGROUPS=[\'identity\',\'math\',\'memory\',\'extraction\',\'familiar\']\ndef evaluate(checkpoint,data,checks,output,device=\'cuda\'):\n    model,metadata=load(checkpoint);model.to(device);model.eval()\n    training,valid=partition_rows(read_jsonl(data),42)\n    _,seen=unique_windows(examples(training,model.config.context,True))\n    validation,_=unique_windows(examples(valid,model.config.context,True),seen)\n    validation=random.Random(49317).sample(validation,min(64,len(validation)))\n    validation.sort(key=lambda pair:len(pair[0]))\n    if not validation:raise ValueError(\'V4 needs independent validation windows\')\n    with torch.no_grad():loss=evaluate_loss(model,validation,4)\n    cases=json.loads(Path(checks).read_text(encoding=\'utf-8\'))[\'cases\']\n    cases=cases+[\n        {\'category\':\'familiar\',\'prompt\':\'What is your name?\',\'expected\':\'Nebula\'},\n        {\'category\':\'familiar\',\'prompt\':\'What is 2 + 2?\',\'number\':4},\n        {\'category\':\'familiar\',\'prompt\':\'My name is Sam. What is my name?\',\'expected\':\'Sam\'}]\n    answers=[]\n    for case in cases:\n        messages=case.get(\'messages\') or [{\'role\':\'user\',\'content\':case[\'prompt\']}]\n        answer=model.reply(with_owner_rules(messages),max_tokens=96,temperature=0)\n        answers.append({\'category\':case[\'category\'],\'messages\':messages,\'reply\':answer,\n                        \'passed\':correct(case,answer),\'healthy\':healthy(answer)})\n    report={\'loss\':loss,\'correct\':sum(x[\'passed\'] for x in answers),\'cases\':len(answers),\n            \'health\':sum(x[\'healthy\'] for x in answers)/len(answers),\n            **{group:sum(x[\'passed\'] for x in answers if x[\'category\']==group)/sum(x[\'category\']==group for x in answers) for group in GROUPS}}\n    for group in {x[\'category\'] for x in answers}-set(GROUPS):\n        group_answers=[x for x in answers if x[\'category\']==group]\n        report[group]=sum(x[\'passed\'] for x in group_answers)/len(group_answers)\n    Path(output).write_text(json.dumps({\'report\':report,\'answers\':answers,\'metadata\':metadata},indent=2),encoding=\'utf-8\')\n    del model\n    if device==\'cuda\':torch.cuda.empty_cache()\n    print(json.dumps({\'v4Evaluation\':report,\'checkpoint\':Path(checkpoint).name}),flush=True)\n    return report\n\ndef improves(base,candidate):\n    import math\n    keys=[\'loss\',\'health\',\'correct\',*GROUPS]\n    if any(not math.isfinite(value.get(key,float(\'nan\'))) for value in [base,candidate] for key in keys):return False\n    if any(candidate[k]+1e-9<base[k] for k in GROUPS) or candidate[\'health\']<base[\'health\']:return False\n    return candidate[\'loss\']<=base[\'loss\']*1.03 and (\n        candidate[\'correct\']>base[\'correct\'] or\n        candidate[\'correct\']==base[\'correct\'] and candidate[\'loss\']<base[\'loss\']*.99)\n\ndef focus_for(report,rejections):\n    if report[\'health\']<.8:return \'language\'\n    groups=sorted(GROUPS[:4],key=lambda group:report[group])\n    focus=groups[rejections%len(groups)]\n    return \'memory\' if focus==\'extraction\' else focus\n', 'model/v4_store.py': '"""Private experiment snapshots, committed atomically with a resume manifest."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\nfrom research_client import ResearchError,digests\n\nclass Store:\n    def __init__(self,repo,token,run):\n        if not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',repo) or not re.fullmatch(r\'[a-zA-Z0-9_-]{1,64}\',run) or not token.startswith(\'hf_\'):\n            raise ResearchError(\'V4_STORAGE_CONFIGURATION_MISSING\')\n        from huggingface_hub import HfApi\n        self.repo=repo;self.token=token;self.prefix=\'experiments/\'+run+\'/\'\n        self.api=HfApi(endpoint=\'https://huggingface.co\',token=token)\n        self.revision=self.info().sha\n    def info(self):\n        try:value=self.api.model_info(self.repo)\n        except Exception:raise ResearchError(\'V4_STORAGE_ACCESS_FAILED\') from None\n        if value.private is not True:raise ResearchError(\'HF_REPOSITORY_MUST_BE_PRIVATE\')\n        return value\n    def state(self,manifest_name=\'state.json\'):\n        if manifest_name not in [\'state.json\',\'prepared.json\']:raise ResearchError(\'V4_INVALID_FILE_REFERENCE\')\n        from huggingface_hub import hf_hub_download\n        from huggingface_hub.errors import EntryNotFoundError\n        try:\n            path=hf_hub_download(self.repo,self.prefix+manifest_name,revision=self.revision,token=self.token)\n            return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n        except EntryNotFoundError:return None\n        except Exception:raise ResearchError(\'V4_RESUME_MANIFEST_UNREADABLE\') from None\n    def download(self,file,destination):\n        from huggingface_hub import hf_hub_download\n        if not file[\'path\'].startswith(self.prefix) or \'..\' in file[\'path\'].split(\'/\') or not 0<file[\'bytes\']<=256*1024*1024:\n            raise ResearchError(\'V4_INVALID_FILE_REFERENCE\')\n        destination=Path(destination);destination.parent.mkdir(parents=True,exist_ok=True)\n        if destination.exists() and destination.stat().st_size==file[\'bytes\'] and digests(destination)[0]==file[\'sha256\']:return destination\n        try:cached=Path(hf_hub_download(self.repo,file[\'path\'],revision=self.revision,token=self.token))\n        except Exception:raise ResearchError(\'V4_SNAPSHOT_DOWNLOAD_FAILED\') from None\n        if cached.stat().st_size!=file[\'bytes\'] or digests(cached)[0]!=file[\'sha256\']:raise ResearchError(\'V4_CHECKSUM_MISMATCH\')\n        temporary=destination.with_suffix(destination.suffix+\'.download-tmp\');shutil.copyfile(cached,temporary);temporary.replace(destination)\n        return destination\n    def publish(self,state,files,project,manifest_name=\'state.json\'):\n        if manifest_name not in [\'state.json\',\'prepared.json\']:raise ResearchError(\'V4_INVALID_FILE_REFERENCE\')\n        from huggingface_hub import CommitOperationAdd\n        self.info();operations=[];state=json.loads(json.dumps(state))\n        for key,(path,name) in files.items():\n            path=Path(path)\n            if \'..\' in name.split(\'/\') or name.startswith(\'/\') or \'\\\\\' in name:raise ResearchError(\'V4_INVALID_FILE_REFERENCE\')\n            if not 0<path.stat().st_size<=256*1024*1024:raise ResearchError(\'V4_SNAPSHOT_TOO_LARGE\')\n            remote=self.prefix+name\n            state[key]={\'path\':remote,\'bytes\':path.stat().st_size,\'sha256\':digests(path)[0]}\n            operations.append(CommitOperationAdd(path_in_repo=remote,path_or_fileobj=str(path)))\n        manifest=Path(project)/(\'v4-\'+manifest_name);manifest.write_text(json.dumps(state,indent=2),encoding=\'utf-8\')\n        operations.append(CommitOperationAdd(path_in_repo=self.prefix+manifest_name,path_or_fileobj=str(manifest)))\n        try:\n            result=self.api.create_commit(repo_id=self.repo,operations=operations,\n                commit_message=\'Save bounded Nebula V4 experiment\',parent_commit=self.revision)\n        except Exception:raise ResearchError(\'V4_BACKUP_FAILED_OR_CONCURRENT_RUN\') from None\n        self.revision=result.oid\n        print(json.dumps({\'v4BackupSaved\':True,\'round\':state.get(\'round\',0),\'publicWebsiteUpdated\':False}),flush=True)\n        return state\n', 'model/v4_experiment.py': '"""A new, bounded Kaggle experiment; never resets or promotes the paused API campaign."""\nimport argparse\nimport gc\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport time\nimport torch\nfrom dataflow import read_jsonl\nfrom kaggle_migrate import restore_data\nfrom nebula import load\nfrom research_client import Client,ResearchError,digests\nfrom research_worker import configure_rules,launch\nfrom v4_data import VERSION,prepare,public_additions\nfrom v4_eval import evaluate,improves,focus_for\nfrom v4_grow import grow_checkpoint\nfrom v4_store import Store\n\ndef profile(project,checkpoint,data,label):\n    measurements={}\n    for gpus in range(1,min(2,torch.cuda.device_count())+1):\n        try:\n            value=launch(project,checkpoint,project/f\'probe-{label}-{gpus}.pt\',20,gpus,\'balanced\',\n                         benchmark=True,lr=.00001,max_seconds=120,data=data)\n        except ResearchError:\n            print(json.dumps({\'gpuProbeFailed\':label,\'gpus\':gpus}),flush=True);continue\n        measurements[gpus]=value\n    if not measurements:raise ResearchError(\'V4_GPU_PROBE_FAILED\')\n    one=measurements.get(1,{}).get(\'targetsPerSecond\',0);two=measurements.get(2,{}).get(\'targetsPerSecond\',0)\n    gpus=2 if two>one*1.05 else 1\n    if gpus not in measurements:gpus=max(measurements)\n    return {\'gpus\':gpus,**measurements[gpus]}\n\ndef size_model(project,baseline,data):\n    original,metadata=load(baseline);layers=original.config.layers;del original\n    original_profile=profile(project,baseline,data,\'original\')\n    for extra in [4,2]:\n        path=project/f\'grown-{layers+extra}.pt\'\n        info=grow_checkpoint(baseline,path,layers+extra)\n        try:measurement=profile(project,path,data,str(layers+extra))\n        except ResearchError:continue\n        if measurement[\'targetsPerSecond\']>=original_profile[\'targetsPerSecond\']*.45:\n            print(json.dumps({\'selectedLargerModel\':info[\'growth\'],\'gpuProfile\':measurement}),flush=True)\n            return path,measurement,info[\'growth\']\n    print(json.dumps({\'keptOriginalSize\':True,\'reason\':\'larger_profiles_failed_or_too_slow\'}),flush=True)\n    return baseline,original_profile,{\'originalLayers\':layers,\'layers\':layers,\'initialFunctionPreserved\':True}\n\ndef main(argv=None):\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--project\',type=Path,required=True)\n    parser.add_argument(\'--run\',default=\'nebula-v4\')\n    parser.add_argument(\'--hours\',type=float,default=6)\n    parser.add_argument(\'--rounds\',type=int,default=24)\n    parser.add_argument(\'--steps\',type=int,default=500)\n    parser.add_argument(\'--public-rows\',type=int,default=1000)\n    args=parser.parse_args(argv)\n    if not 0<args.hours<=8 or not 1<=args.rounds<=48 or not 100<=args.steps<=2000 or not 0<=args.public_rows<=2000:\n        parser.error(\'Use up to 8 hours, 1-48 rounds, 100-2000 steps, and 0-2000 public rows\')\n    if not torch.cuda.is_available():raise ResearchError(\'GPU_REQUIRED\')\n    torch.set_num_threads(2);project=args.project.resolve();project.mkdir(parents=True,exist_ok=True)\n    deadline=time.monotonic()+args.hours*3600\n    store=Store(os.environ.get(\'NEBULA_HF_REPO\',\'\'),os.environ.get(\'NEBULA_HF_TOKEN\',\'\'),args.run)\n    state=store.state()\n    if state:\n        if state.get(\'format\')!=VERSION:raise ResearchError(\'V4_RESUME_VERSION_MISMATCH_USE_NEW_RUN_NAME\')\n        if state.get(\'status\')==\'paused\':\n            print(\'This experiment paused after three rejected attempts. Review saved reports before a new experiment.\',flush=True);return\n        backup=store.download(state[\'sourceData\'],project/\'source-data.pt\');restore_data(backup,project);configure_rules(project)\n        data=store.download(state[\'data\'],project/\'model/data/v4-chat.jsonl\')\n        checks=store.download(state[\'checks\'],project/\'model/data/v4-checks.json\')\n        best=store.download(state[\'best\'],project/\'resume-best.pt\')\n        baseline=store.download(state[\'baseline\'],project/\'preserved-baseline.pt\')\n        print(json.dumps({\'resumedV4\':True,\'completedRounds\':state[\'round\'],\'sourceModelUnchanged\':True}),flush=True)\n    else:\n        client=Client();source=client.call(\'state\')\n        if not source.get(\'head\'):raise ResearchError(\'RESEARCH_SOURCE_NOT_INITIALIZED\')\n        if source.get(\'lease\') and source[\'lease\'][\'expiresAt\']>time.time()*1000:raise ResearchError(\'STOP_OLD_WORKER_AND_WAIT_FOR_LEASE\')\n        backup=client.download(source[\'dataset\'],project/\'source-data.pt\');restore_data(backup,project);configure_rules(project)\n        baseline=client.download(source[\'head\'],project/\'preserved-baseline.pt\')\n        prepared=store.state(\'prepared.json\')\n        if prepared and prepared.get(\'format\')==VERSION and prepared.get(\'sourceDataSha256\')==source[\'files\'][source[\'dataset\']][\'sha256\']:\n            data=store.download(prepared[\'data\'],project/\'model/data/v4-chat.jsonl\')\n            checks=store.download(prepared[\'checks\'],project/\'model/data/v4-checks.json\')\n            path=store.download(prepared[\'dataManifest\'],project/\'model/data/v4-manifest.json\')\n            manifest=json.loads(path.read_text(encoding=\'utf-8\'))\n            print(json.dumps({\'reusedPreparedV4Data\':True,\'conversations\':manifest[\'conversations\']}),flush=True)\n        else:\n            additions=public_additions(project/\'public-additions\',args.public_rows)\n            data,checks,manifest=prepare(project,additions)\n        best,measurement,growth=size_model(project,baseline,data)\n        original_report=evaluate(baseline,data,checks,project/\'original-evaluation.json\')\n        # Growth is checked for exact CPU forward equivalence before the GPU probe.\n        # No large-model training is accepted merely for having more parameters.\n        best_report=evaluate(best,data,checks,project/\'grown-start-evaluation.json\')\n        if any(best_report[key]<original_report[key] for key in [\'identity\',\'math\',\'memory\',\'extraction\',\'familiar\']):\n            raise ResearchError(\'V4_GROWTH_DIAGNOSTIC_REGRESSION\')\n        state={\'format\':VERSION,\'run\':args.run,\'round\':0,\'rejected\':0,\'status\':\'ready\',\n               \'sourceApiHead\':source[\'head\'],\'dataSummary\':manifest,\'profile\':measurement,\'growth\':growth,\n               \'originalReport\':original_report,\'bestReport\':best_report,\'publicAutoUpdate\':False}\n        state=store.publish(state,{\n            \'baseline\':(baseline,\'preserved-baseline.pt\'),\'sourceData\':(backup,\'source-data.pt\'),\n            \'data\':(data,\'data/v4-chat.jsonl\'),\'checks\':(checks,\'data/v4-checks.json\'),\n            \'dataManifest\':(project/\'model/data/v4-manifest.json\',\'data/manifest.json\'),\n            \'originalEvaluation\':(project/\'original-evaluation.json\',\'reports/original.json\'),\n            \'startEvaluation\':(project/\'grown-start-evaluation.json\',\'reports/grown-start.json\'),\n            \'best\':(best,\'initial-model.pt\')},project)\n    completed=0\n    while completed<args.rounds and time.monotonic()<deadline:\n        if state[\'rejected\']>=3:break\n        number=state[\'round\']+1;focus=focus_for(state[\'bestReport\'],state[\'rejected\'])\n        lr=.000025*(.5**state[\'rejected\']);output=project/f\'round-{number:04d}.pt\'\n        print(json.dumps({\'v4Round\':number,\'focus\':focus,\'learningRate\':lr,\'additionalSteps\':args.steps,\n                          \'selectedGpuCount\':state[\'profile\'][\'gpus\']}),flush=True)\n        launch(project,best,output,args.steps,state[\'profile\'][\'gpus\'],focus,lr=lr,\n               max_seconds=max(1,min(3600,deadline-time.monotonic())),data=data,best=True,save_every=100)\n        selected_report=state[\'bestReport\'];selected_path=best;accepted=False;files={}\n        for label,path in [(\'final\',output),(\'lowest-loss\',output.with_suffix(\'.best.pt\'))]:\n            if label==\'lowest-loss\' and digests(path)[0]==digests(output)[0]:continue\n            report_path=project/f\'round-{number:04d}-{label}.json\'\n            report=evaluate(path,data,checks,report_path)\n            files[\'report-\'+label]=(report_path,f\'reports/{number:04d}-{label}.json\')\n            if improves(selected_report,report):selected_report=report;selected_path=path;accepted=True\n        rejected=0 if accepted else state[\'rejected\']+1\n        state={**state,\'round\':number,\'rejected\':rejected,\'status\':\'paused\' if rejected>=3 else \'ready\',\n               \'bestReport\':selected_report,\'lastRoundAccepted\':accepted}\n        files[\'latest\']=(output,f\'checkpoints/{number:04d}-final.pt\')\n        if accepted:files[\'best\']=(selected_path,f\'checkpoints/{number:04d}-selected.pt\')\n        state=store.publish(state,files,project)\n        # Downloads validate committed bytes and also handle a new Kaggle session.\n        best=store.download(state[\'best\'],project/\'resume-best.pt\')\n        print(json.dumps({\'v4RoundAccepted\':accepted,\'rejectedAttempts\':rejected,\'bestReport\':selected_report,\n                          \'apiHeadUnchanged\':True,\'publicWebsiteUpdated\':False}),flush=True)\n        completed+=1;gc.collect();torch.cuda.empty_cache()\n    print(json.dumps({\'v4RunFinished\':True,\'status\':state[\'status\'],\'completedRounds\':state[\'round\'],\n                      \'resumeWithSameRunName\':True,\'publicWebsiteUpdated\':False}),flush=True)\n\nif __name__==\'__main__\':\n    try:main()\n    except ResearchError as error:print(json.dumps({\'v4Stopped\':error.code}),flush=True);raise SystemExit(1)\n', 'model/v5_data.py': '"""Original short conversations and exact arithmetic; preserve V4\'s held-out tasks."""\nimport hashlib\nimport json\nimport random\nfrom pathlib import Path\nfrom dataflow import example_id,eligible,read_jsonl,write_jsonl\nfrom nebula import encode\nfrom rules import with_owner_rules\nfrom v4_data import exercises,forbidden,blocked_questions\n\nVERSION=\'nebula-v5-1\'\nCONVERSATIONS=[\n (\'hi\',\'Hello! How can I help?\'),(\'hello\',\'Hello! What would you like to talk about?\'),\n (\'Good morning.\',\'Good morning! How can I help today?\'),(\'Good evening.\',\'Good evening! What can I help with?\'),\n (\'How are you?\',\'I am ready to help. How are you doing?\'),(\'Thanks!\',\'You are welcome!\'),\n (\'Thank you for helping me.\',\'You are welcome. I am glad I could help.\'),(\'Bye!\',\'Goodbye! Take care.\'),\n (\'What is your name?\',\'I am Nebula AI.\'),(\'Who are you?\',\'I am Nebula AI, an experimental AI assistant.\'),\n (\'Are you a person?\',\'No. I am an AI assistant.\'),(\'Can you make mistakes?\',\'Yes. I can make mistakes. Please check important information.\'),\n (\'What can you do?\',\'I can try to answer questions, write short text, and help you practice simple tasks.\'),\n (\'Do you know everything?\',\'No. My knowledge and abilities are limited.\'),\n (\'Can you help me?\',\'I can try. What do you need help with?\'),(\'I need help.\',\'Tell me what you are working on.\'),\n (\'Can you help me write an email?\',\'Yes. Who is the email for, and what would you like it to say?\'),\n (\'Write a short thank-you note.\',\'Thank you for your help. I really appreciate your time and kindness.\'),\n (\'Write a friendly greeting.\',\'Hello! I hope you are having a good day.\'),\n (\'Write a polite request for help.\',\'Could you please help me with this when you have a moment?\'),\n (\'Write a short apology for being late.\',\'I am sorry I am late. Thank you for waiting.\'),\n (\'Help me plan my day.\',\'List your tasks, choose the most important ones, and leave time for breaks.\'),\n (\'How can I start studying?\',\'Pick one small topic. Read it, practice it, and take a short break.\'),\n (\'I do not understand.\',\'Which part is confusing? I can try to explain it more simply.\'),\n (\'Can you explain that again?\',\'Of course. Tell me which part you would like me to explain.\'),\n (\'What is gravity?\',\'Gravity is the attraction between objects with mass. It keeps us on Earth.\'),\n (\'Why is the sky blue?\',\'Air scatters blue sunlight more strongly than red sunlight, so the daytime sky looks blue.\'),\n (\'What is a planet?\',\'A planet is a large, nearly round object that orbits a star and has cleared its orbital neighborhood.\'),\n (\'What is the Moon?\',\'The Moon is Earth\\\'s natural satellite.\'),\n (\'What is the Sun?\',\'The Sun is the star at the center of our solar system.\'),\n (\'What do plants need to grow?\',\'Most plants need light, water, air, nutrients, and a suitable temperature.\'),\n (\'What is water made of?\',\'A water molecule has two hydrogen atoms and one oxygen atom.\'),\n (\'What is ice?\',\'Ice is solid water.\'),(\'What is a triangle?\',\'A triangle is a shape with three straight sides.\'),\n (\'How many days are in a week?\',\'There are seven days in a week.\'),(\'How many minutes are in an hour?\',\'There are 60 minutes in an hour.\'),\n (\'What is a noun?\',\'A noun is a word that names a person, place, thing, or idea.\'),\n (\'What is a verb?\',\'A verb usually describes an action, event, or state.\'),\n (\'What is an adjective?\',\'An adjective describes a noun.\'),\n (\'What is a computer?\',\'A computer is a machine that follows instructions to process information.\'),\n (\'What is a password?\',\'A password is a secret used to help protect access to an account.\'),\n (\'Should I share my password?\',\'Keep your password private. Do not post it in a chat.\'),\n (\'Tell me a short story.\',\'A puppy lost its ball in the garden. A friend found it under a bench. They played together until sunset.\'),\n (\'Tell me a story about a cat.\',\'A cat watched rain from a warm window. When the sun came out, it curled up and fell asleep.\'),\n (\'Give me a creative project idea.\',\'Make a small illustrated book about a place you would like to visit.\'),\n (\'Help me brainstorm.\',\'What would you like ideas for? Tell me your goal and any limits.\'),\n (\'I am bored.\',\'Would you like a story, a simple game, or a creative project idea?\'),\n (\'I had a difficult day.\',\'That sounds hard. Would you like to talk about what happened?\'),\n (\'I am happy today.\',\'I am glad to hear that! What made your day good?\'),\n (\'What should we talk about?\',\'We could talk about a hobby, a question, or something you want to make.\'),\n]\n\ndef additions():\n    _,held_cases=exercises();blocked=blocked_questions(held_cases);rows=[];seen=set()\n    def add(messages,group,kind):\n        row={\'id\':example_id(messages),\'reviewStatus\':\'approved\',\'splitGroup\':group,\n             \'source\':{\'kind\':\'seed\',\'curriculum\':kind},\'messages\':messages}\n        if row[\'id\'] in seen or forbidden(row,held_cases,blocked) or len(encode(with_owner_rules(messages))[0])>1024:return\n        seen.add(row[\'id\']);rows.append(row)\n    for index,(question,answer) in enumerate(CONVERSATIONS):\n        # Variants stay in one partition, rather than leaking into validation.\n        group=f\'v5-conversation-{index}\'\n        for prefix in [\'\', \'Hi. \', \'Hello. \', \'Please help: \', \'Quick question: \']:\n            for suffix in [\'\', \' Please keep it short.\', \' One sentence is fine.\', \' Please answer clearly.\']:\n                add([{\'role\':\'user\',\'content\':prefix+question+suffix},{\'role\':\'assistant\',\'content\':answer}],group,\'v5-conversation\')\n        add([{\'role\':\'user\',\'content\':question},{\'role\':\'assistant\',\'content\':answer},\n             {\'role\':\'user\',\'content\':\'Thank you.\'},{\'role\':\'assistant\',\'content\':\'You are welcome!\'}],group,\'v5-conversation\')\n    # Exact terminating targets, including familiar 2 + 2 rehearsal. V4\'s held\n    # operand pairs and reserved scenarios remain excluded in every wording.\n    for a in range(32):\n        for b in range(32):\n            for operator,value,verb in [(\'+\',a+b,\'Add\'),(\'-\',a-b,\'Subtract\')]:\n                group=f\'addition-{min(a,b)}-{max(a,b)}\' if operator==\'+\' else f\'v5-subtraction-{a}-{b}\'\n                questions=[f\'What is {a} {operator} {b}?\',f\'Calculate {a} {operator} {b}.\',\n                           f\'{verb} {a} and {b}.\' if operator==\'+\' else f\'Subtract {b} from {a}.\']\n                for q in questions:add([{\'role\':\'user\',\'content\':q},{\'role\':\'assistant\',\'content\':f\'{a} {operator} {b} = {value}.\'}],group,\'v5-exact-math\')\n    return rows\n\ndef prepare(base,output):\n    rows=[];seen=set();new=0\n    for row in read_jsonl(base)+additions():\n        if not eligible(row) or row.get(\'source\',{}).get(\'kind\')==\'user\':continue\n        identifier=example_id(row[\'messages\'])\n        if identifier in seen:continue\n        seen.add(identifier);rows.append(row)\n        if row.get(\'source\',{}).get(\'curriculum\',\'\').startswith(\'v5-\'):new+=1\n    write_jsonl(output,rows)\n    result={\'format\':VERSION,\'conversations\':len(rows),\'newOriginalExamples\':new,\n            \'dataSha256\':hashlib.sha256(Path(output).read_bytes()).hexdigest(),\n            \'rawUserChatsIncluded\':False,\'knownProbeRehearsal\':[\'What is your name?\',\'What is 2 + 2?\'],\n            \'note\':\'Known probes are rehearsal, not unseen tests. Original V4 frozen tasks remain excluded. Public source notices are inherited.\'}\n    Path(output).with_suffix(\'.manifest.json\').write_text(json.dumps(result,indent=2),encoding=\'utf-8\')\n    return result\n', 'model/v5_experiment.py': '"""Resumable learning from V4\'s strongest trial, with separate public releases."""\nimport argparse\nimport gc\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport time\nimport torch\nfrom huggingface_hub import hf_hub_download\nfrom dataflow import read_jsonl\nfrom kaggle_migrate import restore_data\nfrom nebula import load,save\nfrom research_client import Client,ResearchError,digests\nfrom research_worker import configure_rules,launch\nfrom v4_eval import evaluate,focus_for,GROUPS\nfrom v4_experiment import profile\nfrom v4_store import Store\nfrom v5_data import VERSION,prepare\n\nSOURCE_REVISION=\'916b0eaed037df63cdd0add74f10452ce394aaf1\'\n\ndef research_improves(base,candidate):\n    keys=[\'loss\',\'health\',\'correct\',\'cases\',*GROUPS]\n    if any(not math.isfinite(r.get(k,float(\'nan\'))) for r in [base,candidate] for k in keys):return False\n    if base[\'cases\']!=candidate[\'cases\'] or candidate[\'health\']<max(.90,base[\'health\']-.03):return False\n    if candidate[\'loss\']>base[\'loss\']*1.08:return False\n    groups=GROUPS[:4]+([\'conversation\'] if \'conversation\' in base else [])\n    if any(not math.isfinite(r.get(k,float(\'nan\'))) for r in [base,candidate] for k in groups):return False\n    macro=lambda r:sum(r[k] for k in groups)/len(groups)\n    # A research head may trade a few diagnostic cases for broader gains. A\n    # public release is pinned separately and is never changed by this rule.\n    return (candidate[\'correct\']>base[\'correct\'] and macro(candidate)>=macro(base)-.01) or (\n        candidate[\'correct\']==base[\'correct\'] and macro(candidate)>=macro(base) and candidate[\'loss\']<base[\'loss\']*.99)\n\ndef topic_focus(report,rejected,round_number,topics):\n    # At most one round in four follows opted-in usage; the rest follow errors.\n    counts=topics.get(\'counts\',{}) if isinstance(topics,dict) else {}\n    valid={k:v for k,v in counts.items() if k in [\'identity\',\'math\',\'memory\',\'language\'] and\n           type(v) is int and 0<=v<=10000}\n    if round_number%4==0 and valid and max(valid.values())>=5:return max(valid,key=valid.get)\n    if round_number%5==0 or report.get(\'conversation\',1)<.5 and round_number%2==1:return \'language\'\n    return focus_for(report,rejected)\n\ndef compact(source,destination):\n    model,metadata=load(source);save(model,destination,metadata);del model\n    return destination\n\ndef initial(store,project):\n    # Pin the inspected trial rather than following a mutable latest pointer.\n    def get(name,target):\n        try:path=hf_hub_download(store.repo,\'experiments/nebula-v4/\'+name,revision=SOURCE_REVISION,token=store.token)\n        except Exception:raise ResearchError(\'V5_SOURCE_DOWNLOAD_FAILED\') from None\n        target=project/target;target.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(path,target);return target\n    source=get(\'source-data.pt\',\'source-data.pt\');restore_data(source,project);configure_rules(project)\n    base_data=get(\'data/v4-chat.jsonl\',\'model/data/v4-chat.jsonl\')\n    source_manifest=get(\'data/manifest.json\',\'model/data/v4-source-manifest.json\')\n    checks=get(\'data/v4-checks.json\',\'model/data/v4-checks.json\')\n    cases=json.loads(checks.read_text(encoding=\'utf-8\'))\n    cases[\'cases\'] += [\n        {\'category\':\'conversation\',\'prompt\':\'Could you write a brief note thanking someone?\',\'expected\':\'thank\'},\n        {\'category\':\'conversation\',\'prompt\':\'What do you say when someone says thank you?\',\'expected\':\'welcome\'},\n        {\'category\':\'conversation\',\'prompt\':\'Can you tell me whether you are human?\',\'expected\':\'AI\'},\n        {\'category\':\'conversation\',\'prompt\':\'Help me draft an email to someone.\',\'expected\':\'who\'},\n        {\'category\':\'conversation\',\'prompt\':\'Please explain what ice is.\',\'expected\':\'water\'},\n        {\'category\':\'conversation\',\'prompt\':\'How long is one hour in minutes?\',\'number\':60}]\n    checks=project/\'model/data/v5-checks.json\';checks.write_text(json.dumps(cases,indent=2),encoding=\'utf-8\')\n    baseline=get(\'checkpoints/0001-final.pt\',\'preserved-v4-strongest.pt\')\n    data=project/\'model/data/v5-chat.jsonl\';summary=prepare(base_data,data)\n    measurement=profile(project,baseline,data,\'v5\')\n    report=evaluate(baseline,base_data,checks,project/\'baseline-report.json\')\n    state={\'format\':VERSION,\'round\':0,\'rejected\':0,\'bestReport\':report,\'sourceRevision\':SOURCE_REVISION,\n           \'publicAutoUpdate\':False,\'profile\':measurement,\'dataSummary\':summary,\'status\':\'ready\',\n           \'remoteWeightSnapshotsThisSession\':0}\n    return store.publish(state,{\'sourceData\':(source,\'source-data.pt\'),\'baseline\':(baseline,\'preserved-v4-strongest.pt\'),\n       \'evaluationData\':(base_data,\'data/v4-chat.jsonl\'),\'checks\':(checks,\'data/v5-checks.json\'),\n       \'data\':(data,\'data/v5-chat.jsonl\'),\'dataManifest\':(data.with_suffix(\'.manifest.json\'),\'data/manifest.json\'),\n       \'sourceManifest\':(source_manifest,\'data/source-manifest.json\'),\n       \'baselineReport\':(project/\'baseline-report.json\',\'reports/baseline.json\'),\n       \'best\':(compact(baseline,project/\'initial-weights.pt\'),\'best.pt\')},project)\n\ndef main(argv=None):\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--project\',type=Path,required=True);parser.add_argument(\'--run\',default=\'nebula-v5\')\n    parser.add_argument(\'--hours\',type=float,default=8);parser.add_argument(\'--steps\',type=int,default=500)\n    parser.add_argument(\'--rounds\',type=int,default=100)\n    args=parser.parse_args(argv)\n    if not 0<args.hours<=8 or not 100<=args.steps<=2000 or not 1<=args.rounds<=100:parser.error(\'Use 0-8 hours, 100-2000 updates per round, and 1-100 rounds\')\n    if not torch.cuda.is_available():raise ResearchError(\'GPU_REQUIRED\')\n    project=args.project.resolve();project.mkdir(parents=True,exist_ok=True);torch.set_num_threads(2)\n    store=Store(os.environ.get(\'NEBULA_HF_REPO\',\'\'),os.environ.get(\'NEBULA_HF_TOKEN\',\'\'),args.run)\n    state=store.state()\n    if state and state.get(\'format\')!=VERSION:raise ResearchError(\'V5_VERSION_MISMATCH\')\n    if not state:state=initial(store,project)\n    source=store.download(state[\'sourceData\'],project/\'source-data.pt\');restore_data(source,project);configure_rules(project)\n    data=store.download(state[\'data\'],project/\'model/data/v5-chat.jsonl\')\n    evaluation_data=store.download(state[\'evaluationData\'],project/\'model/data/v4-chat.jsonl\')\n    checks=store.download(state[\'checks\'],project/\'model/data/v4-checks.json\')\n    best=store.download(state[\'best\'],project/\'resume-best.pt\')\n    print(json.dumps({\'resumedV5\':True,\'completedRounds\':state[\'round\'],\'bestReport\':state[\'bestReport\'],\n                      \'publicAutoUpdate\':False,\'newData\':state[\'dataSummary\']}),flush=True)\n    deadline=time.monotonic()+args.hours*3600;client=Client();saved=0\n    try:\n        while time.monotonic()<deadline and saved<args.rounds:\n            number=state[\'round\']+1;topics={}\n            try:\n                topics=client.call(\'curriculum\')\n                if topics.get(\'format\')!=\'nebula-topics-1\' or topics.get(\'rawChatsExported\') is not False:topics={}\n            except ResearchError:\n                print(json.dumps({\'chatTopicFeedUnavailable\':True,\'verifiedPracticeContinues\':True}),flush=True)\n            focus=topic_focus(state[\'bestReport\'],state[\'rejected\'],number,topics)\n            # Cycle lower rates and focus on plateaus without discarding saved gains.\n            lr=[.00002,.00001,.000005][min(2,state[\'rejected\']//3)]\n            output=project/\'round.pt\'\n            print(json.dumps({\'v5Round\':number,\'focus\':focus,\'learningRate\':lr,\'additionalSteps\':args.steps,\n                              \'gpus\':state[\'profile\'][\'gpus\'],\'chatTopicFeedAvailable\':bool(topics),\n                              \'chatGuidedRound\':bool(number%4==0 and any(type(v) is int and 5<=v<=10000 for k,v in topics.get(\'counts\',{}).items() if k in [\'identity\',\'math\',\'memory\',\'language\']))}),flush=True)\n            launch(project,best,output,args.steps,state[\'profile\'][\'gpus\'],focus,lr=lr,data=data,best=True,\n                   save_every=100,sampling_seed=(20261017+number)%(2**31),max_seconds=max(1,min(3600,deadline-time.monotonic())))\n            selected=best;report=state[\'bestReport\'];accepted=False;files={}\n            for label,path in [(\'final\',output),(\'lowest-loss\',output.with_suffix(\'.best.pt\'))]:\n                if label==\'lowest-loss\' and digests(path)[0]==digests(output)[0]:continue\n                report_file=project/f\'{label}-report.json\'\n                candidate=evaluate(path,evaluation_data,checks,report_file)\n                files[\'report-\'+label]=(report_file,f\'reports/{number:04d}-{label}.json\')\n                if research_improves(report,candidate):selected=path;report=candidate;accepted=True\n            state={**state,\'round\':number,\'rejected\':0 if accepted else state[\'rejected\']+1,\n                   \'bestReport\':report,\'lastAccepted\':accepted,\'status\':\'ready\'}\n            # Upload weights only: ~33 MB per latest snapshot, no private chats or\n            # large optimizer copies. A restart intentionally starts fresh AdamW.\n            files[\'latest\']=(compact(output,project/\'latest-weights.pt\'),\'latest.pt\')\n            if accepted:files[\'best\']=(compact(selected,project/\'best-weights.pt\'),\'best.pt\')\n            state=store.publish(state,files,project);saved+=1\n            best=store.download(state[\'best\'],project/\'resume-best.pt\')\n            print(json.dumps({\'v5RoundSaved\':number,\'acceptedResearchHead\':accepted,\'bestCorrect\':report[\'correct\'],\n                 \'cases\':report[\'cases\'],\'publicWebsiteUpdated\':False,\'checkpointBackupConfirmed\':True}),flush=True)\n            gc.collect();torch.cuda.empty_cache()\n            stop=project/\'STOP_AFTER_ROUND\'\n            if stop.exists():print(\'Stop file found; the last round is backed up.\');break\n    except KeyboardInterrupt:\n        print(\'Stopped. Resume the last confirmed private backup with the same run name.\',flush=True)\n    print(json.dumps({\'v5SessionFinished\':True,\'savedRound\':state[\'round\'],\'resumeWithSameRunName\':True,\n                      \'publicWebsiteUpdated\':False,\'reason\':\'stop_or_session_budget\'}),flush=True)\n\nif __name__==\'__main__\':\n    try:main()\n    except ResearchError as error:print(json.dumps({\'v5Stopped\':error.code}),flush=True);raise SystemExit(1)\n'}
for name,content in sources.items():
    target=project/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_text(content,encoding='utf-8')
sys.path.insert(0,str(project/'model'))
for name in sources:
    if name.endswith('.py'):sys.modules.pop(Path(name).stem,None)
from v5_experiment import main
main(['--project',str(project),'--run',RUN_NAME,'--hours',str(HOURS),'--rounds',str(MAX_ROUNDS),'--steps',str(STEPS_PER_ROUND)])


## Stop and resume

Use Kaggle's stop button to stop. The last confirmed remote backup remains available. For a graceful stop, create `/kaggle/working/nebula-v5/api/STOP_AFTER_ROUND` from a second cell and wait for the active round to finish. Remove that file before starting again in the same runtime. After a runtime reset, import this notebook, enable the same secrets, and Run all with `RUN_NAME='nebula-v5'`. It resumes the saved best model and retains the latest trial for inspection.

The public website uses the deliberately pinned experimental V4 release packaged with the API. V5 research does not replace it automatically. Inspect actual replies before releasing later checkpoints. Improvements are possible, not guaranteed; an 8.2M-parameter model has substantial limits.
